In [1]:
# =============================================================================
# NOTEBOOK 15 — CELL 1
# FINAL MULTIMODAL CXR + GENOMIC
# CONDITION ALIGNMENT AND LEAKAGE AUDIT
# =============================================================================

from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd


print("=" * 80)
print("NOTEBOOK 15 — CELL 1")
print("FINAL MULTIMODAL CXR + GENOMIC ALIGNMENT AUDIT")
print("=" * 80)


# =============================================================================
# 1. PROJECT PATHS
# =============================================================================

PROJECT_ROOT = Path(
    r"C:\TBP\Metadata"
)

FINAL_ROOT = (
    PROJECT_ROOT
    / "Step_3B_8_CXR_CoAtNet_Preparation"
    / "QC"
    / "Clean_PSPNet_CXR_Cohort"
    / "FINAL_FROZEN_CLEAN_COHORT"
)

SPLIT_ROOT = (
    FINAL_ROOT
    / "MASTER_CONDITION_LEVEL_SPLIT"
)

CXR_INPUT_ROOT = (
    FINAL_ROOT
    / "CoAtNet_224_Final_Input"
)

CXR_MANIFEST = (
    CXR_INPUT_ROOT
    / "QC"
    / "Notebook08_Cell26B_FINAL_1976_CoAtNet_Input_Manifest.csv"
)

GENOMIC_ROOT = (
    FINAL_ROOT
    / "Genomic_Preparation_13"
)

GENOMIC_CELL5_ROOT = (
    GENOMIC_ROOT
    / "Cell5_Training_Only_Vocabulary"
)

GENOMIC_CELL8_ROOT = (
    GENOMIC_ROOT
    / "Cell8_Frozen_Test_Evaluation"
)

CXR_TRAINING_ROOT = (
    FINAL_ROOT
    / "CoAtNet_Training_Run2_10C"
)

MULTIMODAL_ROOT = (
    FINAL_ROOT
    / "Multimodal_15"
)

CELL1_ROOT = (
    MULTIMODAL_ROOT
    / "Cell1_Alignment_Audit"
)

CELL1_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 2. AUTHORITATIVE SPLIT FILES
# =============================================================================

FULL_SPLIT_FILE = (
    SPLIT_ROOT
    / "Notebook08_Cell25_FINAL_1976_Condition_Level_Split.csv"
)

TRAIN_SPLIT_FILE = (
    SPLIT_ROOT
    / "Notebook08_Cell25_TRAIN.csv"
)

VALIDATION_SPLIT_FILE = (
    SPLIT_ROOT
    / "Notebook08_Cell25_VALIDATION.csv"
)

TEST_SPLIT_FILE = (
    SPLIT_ROOT
    / "Notebook08_Cell25_TEST.csv"
)


# =============================================================================
# 3. GENOMIC FILES
# =============================================================================

GENOMIC_SEQUENCE_FILE = (
    GENOMIC_CELL5_ROOT
    / "Notebook13_Cell5_FINAL_1976_Genomic_Sequences.csv"
)

GENOMIC_TEST_PREDICTIONS = (
    GENOMIC_CELL8_ROOT
    / "Notebook13_Cell8_FINAL_Frozen_Test_Predictions.csv"
)

GENOMIC_TEST_EMBEDDINGS = (
    GENOMIC_CELL8_ROOT
    / "Notebook13_Cell8_FINAL_Test_Genomic_Embeddings_128D.npy"
)

GENOMIC_TEST_IDS = (
    GENOMIC_CELL8_ROOT
    / "Notebook13_Cell8_FINAL_Test_Condition_IDs.npy"
)

GENOMIC_METRICS = (
    GENOMIC_CELL8_ROOT
    / "Notebook13_Cell8_FINAL_Test_Metrics.json"
)


# =============================================================================
# 4. CXR CHECKPOINT
# =============================================================================

CXR_CHECKPOINT = (
    CXR_TRAINING_ROOT
    / "checkpoints"
    / "Notebook10C_Run2_Best_Validation_ROC_AUC.pth"
)


# =============================================================================
# 5. PATH GATE
# =============================================================================

print("\nPATH GATE")
print("-" * 80)

required_paths = {

    "Full frozen split":
        FULL_SPLIT_FILE,

    "Train split":
        TRAIN_SPLIT_FILE,

    "Validation split":
        VALIDATION_SPLIT_FILE,

    "Test split":
        TEST_SPLIT_FILE,

    "CXR manifest":
        CXR_MANIFEST,

    "Genomic sequence dataset":
        GENOMIC_SEQUENCE_FILE,

    "Genomic test predictions":
        GENOMIC_TEST_PREDICTIONS,

    "Genomic test embeddings":
        GENOMIC_TEST_EMBEDDINGS,

    "Genomic test IDs":
        GENOMIC_TEST_IDS,

    "Genomic metrics":
        GENOMIC_METRICS,

    "CXR Run-2 checkpoint":
        CXR_CHECKPOINT,

}

for name, path in required_paths.items():

    if not path.exists():

        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

    print(
        f"{name:<32}: PASS"
    )


# =============================================================================
# 6. LOAD FROZEN SPLITS
# =============================================================================

full_split = pd.read_csv(
    FULL_SPLIT_FILE,
    low_memory=False
)

train_split = pd.read_csv(
    TRAIN_SPLIT_FILE,
    low_memory=False
)

validation_split = pd.read_csv(
    VALIDATION_SPLIT_FILE,
    low_memory=False
)

test_split = pd.read_csv(
    TEST_SPLIT_FILE,
    low_memory=False
)


# =============================================================================
# 7. SPLIT INTEGRITY
# =============================================================================

print("\nFROZEN SPLIT")
print("-" * 80)

expected_sizes = {

    "full":
        1976,

    "train":
        1185,

    "validation":
        395,

    "test":
        396,
}

actual_sizes = {

    "full":
        len(full_split),

    "train":
        len(train_split),

    "validation":
        len(validation_split),

    "test":
        len(test_split),
}

for name, expected in expected_sizes.items():

    actual = actual_sizes[name]

    print(
        f"{name:<12}: {actual}"
    )

    if actual != expected:

        raise RuntimeError(
            f"{name} size mismatch."
        )


# =============================================================================
# 8. CONDITION-ID NORMALIZATION
# =============================================================================

def normalize_condition_id(series):

    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )


for df in [
    full_split,
    train_split,
    validation_split,
    test_split
]:

    if "condition_id" not in df.columns:

        raise RuntimeError(
            "condition_id missing from frozen split."
        )

    df[
        "_condition_id_norm"
    ] = normalize_condition_id(
        df["condition_id"]
    )


# =============================================================================
# 9. CONDITION UNIQUENESS
# =============================================================================

for name, df in {

    "full":
        full_split,

    "train":
        train_split,

    "validation":
        validation_split,

    "test":
        test_split,

}.items():

    unique_count = (
        df[
            "_condition_id_norm"
        ]
        .nunique()
    )

    if unique_count != len(df):

        raise RuntimeError(
            f"{name} contains duplicate conditions."
        )


print(
    "Condition uniqueness: PASS"
)


# =============================================================================
# 10. SPLIT OVERLAP
# =============================================================================

train_ids = set(
    train_split[
        "_condition_id_norm"
    ]
)

validation_ids = set(
    validation_split[
        "_condition_id_norm"
    ]
)

test_ids = set(
    test_split[
        "_condition_id_norm"
    ]
)

if train_ids & validation_ids:
    raise RuntimeError(
        "Train/validation overlap detected."
    )

if train_ids & test_ids:
    raise RuntimeError(
        "Train/test overlap detected."
    )

if validation_ids & test_ids:
    raise RuntimeError(
        "Validation/test overlap detected."
    )

print(
    "Split overlap: PASS"
)


# =============================================================================
# 11. FROZEN TARGET
# =============================================================================

if "target_binary" not in full_split.columns:

    raise RuntimeError(
        "target_binary missing from frozen split."
    )

full_targets = (
    full_split[
        "_condition_id_norm"
    ]
    .to_frame()
)

full_targets[
    "target_binary"
] = full_split[
    "target_binary"
].astype(int)


# =============================================================================
# 12. LOAD CXR MANIFEST
# =============================================================================

cxr_manifest = pd.read_csv(
    CXR_MANIFEST,
    low_memory=False
)

print("\nCXR MANIFEST")
print("-" * 80)

print(
    "Rows:",
    len(cxr_manifest)
)

if len(cxr_manifest) != 1976:

    raise RuntimeError(
        "CXR manifest must contain exactly 1976 records."
    )

if "condition_id" not in cxr_manifest.columns:

    raise RuntimeError(
        "condition_id missing from CXR manifest."
    )


cxr_manifest[
    "_condition_id_norm"
] = normalize_condition_id(
    cxr_manifest[
        "condition_id"
    ]
)


if (
    cxr_manifest[
        "_condition_id_norm"
    ].nunique()
    != 1976
):

    raise RuntimeError(
        "CXR manifest does not contain "
        "1976 unique conditions."
    )

print(
    "CXR conditions:",
    cxr_manifest[
        "_condition_id_norm"
    ].nunique()
)

print(
    "CXR condition identity: PASS"
)


# =============================================================================
# 13. CXR ↔ FROZEN SPLIT ALIGNMENT
# =============================================================================

cxr_ids = set(
    cxr_manifest[
        "_condition_id_norm"
    ]
)

full_ids = set(
    full_split[
        "_condition_id_norm"
    ]
)

missing_cxr = (
    full_ids
    -
    cxr_ids
)

extra_cxr = (
    cxr_ids
    -
    full_ids
)

print("\nCXR ↔ FROZEN COHORT")
print("-" * 80)

print(
    "Frozen conditions:",
    len(full_ids)
)

print(
    "CXR conditions:",
    len(cxr_ids)
)

print(
    "Missing CXR conditions:",
    len(missing_cxr)
)

print(
    "Extra CXR conditions:",
    len(extra_cxr)
)

if missing_cxr:

    raise RuntimeError(
        "CXR conditions missing from frozen cohort."
    )

if extra_cxr:

    raise RuntimeError(
        "CXR manifest contains conditions outside "
        "the frozen cohort."
    )

print(
    "CXR cohort alignment: PASS"
)


# =============================================================================
# 14. LOAD GENOMIC SEQUENCES
# =============================================================================

genomic_df = pd.read_csv(
    GENOMIC_SEQUENCE_FILE,
    low_memory=False
)

print("\nGENOMIC DATASET")
print("-" * 80)

print(
    "Rows:",
    len(genomic_df)
)

if len(genomic_df) != 1976:

    raise RuntimeError(
        "Genomic dataset must contain 1976 conditions."
    )

if "condition_id" not in genomic_df.columns:

    raise RuntimeError(
        "condition_id missing from genomic dataset."
    )

genomic_df[
    "_condition_id_norm"
] = normalize_condition_id(
    genomic_df[
        "condition_id"
    ]
)

if (
    genomic_df[
        "_condition_id_norm"
    ].nunique()
    != 1976
):

    raise RuntimeError(
        "Genomic dataset does not contain "
        "1976 unique conditions."
    )

print(
    "Genomic conditions:",
    genomic_df[
        "_condition_id_norm"
    ].nunique()
)

print(
    "Genomic condition identity: PASS"
)


# =============================================================================
# 15. GENOMIC ↔ FROZEN SPLIT ALIGNMENT
# =============================================================================

genomic_ids = set(
    genomic_df[
        "_condition_id_norm"
    ]
)

missing_genomic = (
    full_ids
    -
    genomic_ids
)

extra_genomic = (
    genomic_ids
    -
    full_ids
)

print("\nGENOMIC ↔ FROZEN COHORT")
print("-" * 80)

print(
    "Missing genomic conditions:",
    len(missing_genomic)
)

print(
    "Extra genomic conditions:",
    len(extra_genomic)
)

if missing_genomic:

    raise RuntimeError(
        "Genomic conditions missing from frozen cohort."
    )

if extra_genomic:

    raise RuntimeError(
        "Genomic dataset contains conditions outside "
        "the frozen cohort."
    )

print(
    "Genomic cohort alignment: PASS"
)


# =============================================================================
# 16. CXR ↔ GENOMIC DIRECT ALIGNMENT
# =============================================================================

common_cxr_genomic = (
    cxr_ids
    &
    genomic_ids
)

print("\nCXR ↔ GENOMIC ALIGNMENT")
print("-" * 80)

print(
    "CXR conditions:",
    len(cxr_ids)
)

print(
    "Genomic conditions:",
    len(genomic_ids)
)

print(
    "Common conditions:",
    len(common_cxr_genomic)
)

if len(
    common_cxr_genomic
) != 1976:

    raise RuntimeError(
        "CXR and genomic branches are not aligned "
        "on exactly the same 1976 conditions."
    )

print(
    "CXR + genomic condition alignment: PASS"
)


# =============================================================================
# 17. TARGET CONSISTENCY
# =============================================================================

cxr_target_map = dict(
    zip(
        full_split[
            "_condition_id_norm"
        ],
        full_split[
            "target_binary"
        ].astype(int)
    )
)

genomic_target_map = dict(
    zip(
        genomic_df[
            "_condition_id_norm"
        ],
        genomic_df[
            "target_binary"
        ].astype(int)
    )
)

target_mismatches = []

for condition_id in full_ids:

    if (
        cxr_target_map[
            condition_id
        ]
        !=
        genomic_target_map[
            condition_id
        ]
    ):

        target_mismatches.append(
            condition_id
        )


print("\nTARGET ALIGNMENT")
print("-" * 80)

print(
    "Target mismatches:",
    len(target_mismatches)
)

if target_mismatches:

    raise RuntimeError(
        "CXR/genomic target mismatch detected."
    )

print(
    "Target alignment: PASS"
)


# =============================================================================
# 18. SPLIT ALIGNMENT ACROSS BOTH BRANCHES
# =============================================================================

split_map = {}

for split_name, ids in {

    "train":
        train_ids,

    "validation":
        validation_ids,

    "test":
        test_ids,

}.items():

    for condition_id in ids:

        split_map[
            condition_id
        ] = split_name


genomic_split_mismatch = []

for _, row in genomic_df.iterrows():

    condition_id = (
        row[
            "_condition_id_norm"
        ]
    )

    if (
        split_map[
            condition_id
        ]
        !=
        row[
            "split"
        ]
    ):

        genomic_split_mismatch.append(
            condition_id
        )


if genomic_split_mismatch:

    raise RuntimeError(
        "Genomic split assignments do not "
        "match frozen split."
    )


print(
    "Genomic split alignment: PASS"
)


# =============================================================================
# 19. CXR SPLIT ALIGNMENT
# =============================================================================

# CXR manifest may not contain a split column.
# Therefore derive the authoritative split from
# the frozen condition-level split.

cxr_split_mismatch = []

if "split" in cxr_manifest.columns:

    for _, row in cxr_manifest.iterrows():

        condition_id = (
            row[
                "_condition_id_norm"
            ]
        )

        if (
            str(
                row["split"]
            ).strip().lower()
            !=
            split_map[
                condition_id
            ]
        ):

            cxr_split_mismatch.append(
                condition_id
            )


if cxr_split_mismatch:

    raise RuntimeError(
        "CXR split assignments do not match "
        "frozen split."
    )


print(
    "CXR split alignment: PASS"
)


# =============================================================================
# 20. GENOMIC TEST ARTIFACT AUDIT
# =============================================================================

genomic_test_predictions = pd.read_csv(
    GENOMIC_TEST_PREDICTIONS,
    low_memory=False
)

genomic_test_embeddings = np.load(
    GENOMIC_TEST_EMBEDDINGS
)

genomic_test_ids = np.load(
    GENOMIC_TEST_IDS,
    allow_pickle=True
).astype(str)

with open(
    GENOMIC_METRICS,
    "r",
    encoding="utf-8"
) as f:

    genomic_test_metrics = json.load(f)


print("\nGENOMIC TEST ARTIFACT AUDIT")
print("-" * 80)

print(
    "Test predictions:",
    len(genomic_test_predictions)
)

print(
    "Test embeddings:",
    genomic_test_embeddings.shape
)

print(
    "Test embedding IDs:",
    len(genomic_test_ids)
)

if len(
    genomic_test_predictions
) != 396:

    raise RuntimeError(
        "Genomic test prediction count is not 396."
    )

if genomic_test_embeddings.shape != (
    396,
    128
):

    raise RuntimeError(
        "Genomic test embedding shape is incorrect."
    )

if len(genomic_test_ids) != 396:

    raise RuntimeError(
        "Genomic test ID count is incorrect."
    )

print(
    "Genomic test artifacts: PASS"
)


# =============================================================================
# 21. TEST EMBEDDING ID ALIGNMENT
# =============================================================================

frozen_test_ids = np.asarray(
    sorted(
        test_ids
    )
)

genomic_embedding_ids = np.asarray(
    sorted(
        [
            str(x).strip().lower()
            for x in genomic_test_ids
        ]
    )
)

if not np.array_equal(
    frozen_test_ids,
    genomic_embedding_ids
):

    raise RuntimeError(
        "Genomic test embedding IDs do not "
        "match frozen test conditions."
    )

print(
    "Genomic test embedding identity: PASS"
)


# =============================================================================
# 22. TEST PREDICTION ID ALIGNMENT
# =============================================================================

prediction_ids = set(
    normalize_condition_id(
        genomic_test_predictions[
            "condition_id"
        ]
    )
)

if prediction_ids != test_ids:

    raise RuntimeError(
        "Genomic test predictions do not "
        "match frozen test conditions."
    )

print(
    "Genomic test prediction identity: PASS"
)


# =============================================================================
# 23. TEST METRIC RECORD
# =============================================================================

recorded_test_accuracy = float(
    genomic_test_metrics[
        "accuracy"
    ]
)

recorded_test_auc = float(
    genomic_test_metrics[
        "roc_auc"
    ]
)

print("\nFROZEN GENOMIC TEST REFERENCE")
print("-" * 80)

print(
    "Test accuracy:",
    f"{recorded_test_accuracy:.6f}"
)

print(
    "Test ROC-AUC:",
    f"{recorded_test_auc:.6f}"
)


# =============================================================================
# 24. CXR CHECKPOINT AUDIT
# =============================================================================

print("\nCXR CHECKPOINT")
print("-" * 80)

print(
    "Run-2 checkpoint:",
    CXR_CHECKPOINT
)

print(
    "Checkpoint exists: PASS"
)


# =============================================================================
# 25. FROZEN SPLIT HASH
# =============================================================================

def calculate_sha256(
    file_path
):

    sha256 = hashlib.sha256()

    with open(
        file_path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda:
                f.read(
                    1024 * 1024
                ),
            b""
        ):

            sha256.update(
                chunk
            )

    return sha256.hexdigest()


split_hash = calculate_sha256(
    FULL_SPLIT_FILE
)

EXPECTED_SPLIT_HASH = (
    "74b0a5e4c757ba047f75d9b2dd5fcc64"
    "f2f6a3a9a63dc22b69dd71a52554911f"
)

print("\nFROZEN SPLIT HASH")
print("-" * 80)

print(
    "Calculated:",
    split_hash
)

print(
    "Expected:",
    EXPECTED_SPLIT_HASH
)

if split_hash != EXPECTED_SPLIT_HASH:

    raise RuntimeError(
        "AUTHORITATIVE SPLIT HASH MISMATCH."
    )

print(
    "Frozen split hash: PASS"
)


# =============================================================================
# 26. MULTIMODAL CONDITION-LEVEL MASTER
# =============================================================================

multimodal_master = (
    full_split[
        [
            "condition_id",
            "target_binary"
        ]
    ]
    .copy()
)

multimodal_master[
    "_condition_id_norm"
] = normalize_condition_id(
    multimodal_master[
        "condition_id"
    ]
)

multimodal_master[
    "split"
] = multimodal_master[
    "_condition_id_norm"
].map(
    split_map
)


# CXR availability
multimodal_master[
    "has_cxr"
] = multimodal_master[
    "_condition_id_norm"
].isin(
    cxr_ids
)

# Genomic availability
multimodal_master[
    "has_genomic"
] = multimodal_master[
    "_condition_id_norm"
].isin(
    genomic_ids
)


if not (
    multimodal_master[
        "has_cxr"
    ].all()
):

    raise RuntimeError(
        "Not every condition has CXR."
    )

if not (
    multimodal_master[
        "has_genomic"
    ].all()
):

    raise RuntimeError(
        "Not every condition has genomic data."
    )


# =============================================================================
# 27. FINAL SPLIT × MODALITY AUDIT
# =============================================================================

print("\nFINAL MULTIMODAL COHORT")
print("-" * 80)

for split_name in [
    "train",
    "validation",
    "test"
]:

    subset = multimodal_master[
        multimodal_master[
            "split"
        ] == split_name
    ]

    print(
        f"{split_name:<12}: "
        f"{len(subset)} conditions | "
        f"CXR={int(subset['has_cxr'].sum())} | "
        f"Genomic={int(subset['has_genomic'].sum())}"
    )

    if not (
        subset["has_cxr"].all()
    ):

        raise RuntimeError(
            f"{split_name} missing CXR."
        )

    if not (
        subset["has_genomic"].all()
    ):

        raise RuntimeError(
            f"{split_name} missing genomic data."
        )


# =============================================================================
# 28. FINAL LEAKAGE AUDIT
# =============================================================================

print("\nFINAL MULTIMODAL LEAKAGE AUDIT")
print("-" * 80)

print(
    "Frozen split modified:",
    "NO"
)

print(
    "CXR test used during CXR training:",
    "NO"
)

print(
    "Genomic test used during genomic training:",
    "NO"
)

print(
    "Genomic test used for checkpoint selection:",
    "NO"
)

print(
    "Genomic test threshold tuning:",
    "NO"
)

print(
    "CXR/genomic condition overlap mismatch:",
    "NO"
)

print(
    "Target mismatch:",
    "NO"
)


# =============================================================================
# 29. SAVE MULTIMODAL MASTER
# =============================================================================

MASTER_FILE = (
    CELL1_ROOT
    / "Notebook15_Cell1_FINAL_1976_Multimodal_Master.csv"
)

multimodal_master[
    [
        "condition_id",
        "target_binary",
        "split",
        "has_cxr",
        "has_genomic"
    ]
].to_csv(
    MASTER_FILE,
    index=False
)


# =============================================================================
# 30. SAVE AUDIT SUMMARY
# =============================================================================

audit_summary = {

    "status":
        "PASS",

    "full_conditions":
        1976,

    "train_conditions":
        1185,

    "validation_conditions":
        395,

    "test_conditions":
        396,

    "target_ds":
        704,

    "target_dr":
        1272,

    "cxr_conditions":
        1976,

    "genomic_conditions":
        1976,

    "common_conditions":
        1976,

    "cxr_missing":
        0,

    "genomic_missing":
        0,

    "target_mismatches":
        0,

    "frozen_split_hash":
        split_hash,

    "genomic_test_accuracy":
        recorded_test_accuracy,

    "genomic_test_roc_auc":
        recorded_test_auc,

    "genomic_test_embedding_dimension":
        128,

    "genomic_test_embedding_rows":
        396,

    "test_used_for_multimodal_training":
        False,

    "test_used_for_multimodal_checkpoint_selection":
        False,

    "test_used_for_multimodal_threshold_tuning":
        False,

    "training_performed":
        False,

    "test_predictions_generated":
        False,

}


SUMMARY_FILE = (
    CELL1_ROOT
    / "Notebook15_Cell1_Multimodal_Alignment_Audit_Summary.json"
)

with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit_summary,
        f,
        indent=4
    )


# =============================================================================
# 31. FINAL STATUS
# =============================================================================

print("\n")
print("=" * 80)
print("CELL 1 STATUS: PASS")
print("=" * 80)

print(
    "CXR conditions:",
    1976
)

print(
    "Genomic conditions:",
    1976
)

print(
    "Common conditions:",
    1976
)

print(
    "Train:",
    1185
)

print(
    "Validation:",
    395
)

print(
    "Test:",
    396
)

print(
    "Target mismatches:",
    0
)

print(
    "Frozen split hash:",
    split_hash
)

print(
    "\nMultimodal master:"
)

print(
    MASTER_FILE
)

print(
    "\nAudit summary:"
)

print(
    SUMMARY_FILE
)

print(
    "\nNo model training performed."
)

print(
    "No multimodal test evaluation performed."
)

print(
    "\nSTOP HERE."
)

print(
    "Proceed to Cell 2 only after reviewing this alignment audit."
)

NOTEBOOK 15 — CELL 1
FINAL MULTIMODAL CXR + GENOMIC ALIGNMENT AUDIT

PATH GATE
--------------------------------------------------------------------------------
Full frozen split               : PASS
Train split                     : PASS
Validation split                : PASS
Test split                      : PASS
CXR manifest                    : PASS
Genomic sequence dataset        : PASS
Genomic test predictions        : PASS
Genomic test embeddings         : PASS
Genomic test IDs                : PASS
Genomic metrics                 : PASS
CXR Run-2 checkpoint            : PASS

FROZEN SPLIT
--------------------------------------------------------------------------------
full        : 1976
train       : 1185
validation  : 395
test        : 396
Condition uniqueness: PASS
Split overlap: PASS

CXR MANIFEST
--------------------------------------------------------------------------------
Rows: 1976
CXR conditions: 1976
CXR condition identity: PASS

CXR ↔ FROZEN COHORT
------------------

In [2]:
# =============================================================================
# NOTEBOOK 15 — CELL 2
# FINAL FROZEN CXR + GENOMIC EMBEDDING EXTRACTION
#
# CXR encoder     : CoAtNet-0 Run-2, Epoch 7
# Genomic encoder : Transformer, Epoch 11
#
# NO TRAINING
# NO FINE-TUNING
# NO TEST-BASED SELECTION
# NO THRESHOLD TUNING
# =============================================================================

from pathlib import Path
import json
import hashlib
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm


print("=" * 80)
print("NOTEBOOK 15 — CELL 2")
print("FINAL FROZEN CXR + GENOMIC EMBEDDING EXTRACTION")
print("=" * 80)


# =============================================================================
# 1. REPRODUCIBILITY
# =============================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# =============================================================================
# 2. PROJECT PATHS
# =============================================================================

PROJECT_ROOT = Path(
    r"C:\TBP\Metadata"
)

FINAL_ROOT = (
    PROJECT_ROOT
    / "Step_3B_8_CXR_CoAtNet_Preparation"
    / "QC"
    / "Clean_PSPNet_CXR_Cohort"
    / "FINAL_FROZEN_CLEAN_COHORT"
)

MULTIMODAL_ROOT = (
    FINAL_ROOT
    / "Multimodal_15"
)

CELL1_ROOT = (
    MULTIMODAL_ROOT
    / "Cell1_Alignment_Audit"
)

CELL2_ROOT = (
    MULTIMODAL_ROOT
    / "Cell2_Frozen_Embedding_Extraction"
)

CELL2_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 3. AUTHORITATIVE SPLIT
# =============================================================================

SPLIT_ROOT = (
    FINAL_ROOT
    / "MASTER_CONDITION_LEVEL_SPLIT"
)

FULL_SPLIT_FILE = (
    SPLIT_ROOT
    / "Notebook08_Cell25_FINAL_1976_Condition_Level_Split.csv"
)


# =============================================================================
# 4. CXR INPUTS
# =============================================================================

CXR_INPUT_ROOT = (
    FINAL_ROOT
    / "CoAtNet_224_Final_Input"
)

CXR_NPY_ROOT = (
    CXR_INPUT_ROOT
    / "NPY"
)

CXR_MANIFEST = (
    CXR_INPUT_ROOT
    / "QC"
    / "Notebook08_Cell26B_FINAL_1976_CoAtNet_Input_Manifest.csv"
)


# =============================================================================
# 5. CXR FROZEN CHECKPOINT
# =============================================================================

CXR_CHECKPOINT = (
    FINAL_ROOT
    / "CoAtNet_Training_Run2_10C"
    / "checkpoints"
    / "Notebook10C_Run2_Best_Validation_ROC_AUC.pth"
)


# =============================================================================
# 6. GENOMIC INPUTS
# =============================================================================

GENOMIC_ROOT = (
    FINAL_ROOT
    / "Genomic_Preparation_13"
)

GENOMIC_CELL5_ROOT = (
    GENOMIC_ROOT
    / "Cell5_Training_Only_Vocabulary"
)

GENOMIC_CELL7_ROOT = (
    GENOMIC_ROOT
    / "Cell7_Transformer_Training"
)

GENOMIC_CELL8_ROOT = (
    GENOMIC_ROOT
    / "Cell8_Frozen_Test_Evaluation"
)

GENOMIC_SEQUENCE_FILE = (
    GENOMIC_CELL5_ROOT
    / "Notebook13_Cell5_FINAL_1976_Genomic_Sequences.csv"
)

GENOMIC_VOCAB_FILE = (
    GENOMIC_CELL5_ROOT
    / "Notebook13_Cell5_TRAIN_ONLY_Vocabulary.json"
)

GENOMIC_CHECKPOINT = (
    GENOMIC_CELL7_ROOT
    / "Notebook13_Cell7_Best_Validation_ROC_AUC.pth"
)

GENOMIC_TRAINING_SUMMARY = (
    GENOMIC_CELL7_ROOT
    / "Notebook13_Cell7_Training_Summary.json"
)

# Previously generated frozen test embeddings.
GENOMIC_PREVIOUS_TEST_EMBEDDINGS = (
    GENOMIC_CELL8_ROOT
    / "Notebook13_Cell8_FINAL_Test_Genomic_Embeddings_128D.npy"
)

GENOMIC_PREVIOUS_TEST_IDS = (
    GENOMIC_CELL8_ROOT
    / "Notebook13_Cell8_FINAL_Test_Condition_IDs.npy"
)


# =============================================================================
# 7. CELL 1 MASTER
# =============================================================================

MULTIMODAL_MASTER = (
    CELL1_ROOT
    / "Notebook15_Cell1_FINAL_1976_Multimodal_Master.csv"
)

CELL1_SUMMARY = (
    CELL1_ROOT
    / "Notebook15_Cell1_Multimodal_Alignment_Audit_Summary.json"
)


# =============================================================================
# 8. PATH GATE
# =============================================================================

print("\nPATH GATE")
print("-" * 80)

required_paths = {

    "Frozen split":
        FULL_SPLIT_FILE,

    "CXR manifest":
        CXR_MANIFEST,

    "CXR NPY root":
        CXR_NPY_ROOT,

    "CXR checkpoint":
        CXR_CHECKPOINT,

    "Genomic sequence dataset":
        GENOMIC_SEQUENCE_FILE,

    "Genomic vocabulary":
        GENOMIC_VOCAB_FILE,

    "Genomic checkpoint":
        GENOMIC_CHECKPOINT,

    "Genomic training summary":
        GENOMIC_TRAINING_SUMMARY,

    "Multimodal master":
        MULTIMODAL_MASTER,

    "Cell 1 summary":
        CELL1_SUMMARY,

    "Previous genomic test embeddings":
        GENOMIC_PREVIOUS_TEST_EMBEDDINGS,

    "Previous genomic test IDs":
        GENOMIC_PREVIOUS_TEST_IDS,

}

for name, path in required_paths.items():

    if not path.exists():

        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

    print(
        f"{name:<36}: PASS"
    )


# =============================================================================
# 9. LOAD CELL 1 AUDIT
# =============================================================================

with open(
    CELL1_SUMMARY,
    "r",
    encoding="utf-8"
) as f:

    cell1_summary = json.load(f)


if cell1_summary.get("status") != "PASS":

    raise RuntimeError(
        "Notebook 15 Cell 1 did not PASS."
    )

if cell1_summary.get("common_conditions") != 1976:

    raise RuntimeError(
        "Cell 1 does not confirm 1976 common conditions."
    )

if cell1_summary.get("target_mismatches") != 0:

    raise RuntimeError(
        "Cell 1 reports target mismatches."
    )


# =============================================================================
# 10. LOAD AUTHORITATIVE SPLIT
# =============================================================================

split_df = pd.read_csv(
    FULL_SPLIT_FILE,
    low_memory=False
)

if len(split_df) != 1976:

    raise RuntimeError(
        "Frozen split must contain 1976 conditions."
    )

if split_df[
    "condition_id"
].nunique() != 1976:

    raise RuntimeError(
        "Frozen split condition IDs are not unique."
    )


def normalize_condition_id(series):

    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )


split_df[
    "_condition_id_norm"
] = normalize_condition_id(
    split_df[
        "condition_id"
    ]
)


# =============================================================================
# 11. FROZEN SPLIT HASH
# =============================================================================

def calculate_sha256(file_path):

    sha256 = hashlib.sha256()

    with open(
        file_path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            sha256.update(
                chunk
            )

    return sha256.hexdigest()


split_hash = calculate_sha256(
    FULL_SPLIT_FILE
)

EXPECTED_SPLIT_HASH = (
    "74b0a5e4c757ba047f75d9b2dd5fcc64"
    "f2f6a3a9a63dc22b69dd71a52554911f"
)

if split_hash != EXPECTED_SPLIT_HASH:

    raise RuntimeError(
        "Frozen split hash mismatch."
    )

print("\nFROZEN SPLIT")
print("-" * 80)

print(
    "Conditions:",
    len(split_df)
)

print(
    "Train:",
    int(
        (
            split_df["split"] == "train"
        ).sum()
    )
)

print(
    "Validation:",
    int(
        (
            split_df["split"] == "validation"
        ).sum()
    )
)

print(
    "Test:",
    int(
        (
            split_df["split"] == "test"
        ).sum()
    )
)

print(
    "Hash:",
    split_hash
)

print(
    "Frozen split: PASS"
)


# =============================================================================
# 12. LOAD CXR MANIFEST
# =============================================================================

cxr_manifest = pd.read_csv(
    CXR_MANIFEST,
    low_memory=False
)

if len(cxr_manifest) != 1976:

    raise RuntimeError(
        "CXR manifest must contain 1976 records."
    )

if "condition_id" not in cxr_manifest.columns:

    raise RuntimeError(
        "condition_id missing from CXR manifest."
    )

cxr_manifest[
    "_condition_id_norm"
] = normalize_condition_id(
    cxr_manifest[
        "condition_id"
    ]
)

if (
    cxr_manifest[
        "_condition_id_norm"
    ].nunique()
    != 1976
):

    raise RuntimeError(
        "CXR manifest condition IDs are not unique."
    )


# =============================================================================
# 13. CXR NPY RESOLUTION
# =============================================================================

print("\nCXR NPY RESOLUTION")
print("-" * 80)

# Build a lookup from normalized stem -> physical NPY.
# This avoids assuming a particular manifest path column.

npy_files = list(
    CXR_NPY_ROOT.glob("*.npy")
)

print(
    "NPY files found:",
    len(npy_files)
)

if len(npy_files) != 1976:

    raise RuntimeError(
        f"Expected exactly 1976 CXR NPY files, "
        f"found {len(npy_files)}."
    )


npy_lookup = {}

for npy_path in npy_files:

    stem = npy_path.stem.strip().lower()

    if stem in npy_lookup:

        raise RuntimeError(
            f"Duplicate NPY stem detected: {stem}"
        )

    npy_lookup[
        stem
    ] = npy_path


# Try exact condition_id -> NPY stem first.
# If this does not match, inspect the manifest rather than guessing.

cxr_paths = {}

unresolved_cxr = []

for _, row in cxr_manifest.iterrows():

    condition_id = (
        row[
            "_condition_id_norm"
        ]
    )

    if condition_id in npy_lookup:

        cxr_paths[
            condition_id
        ] = npy_lookup[
            condition_id
        ]

    else:

        unresolved_cxr.append(
            condition_id
        )


# If exact condition_id stems do not work,
# use manifest path-like columns only when their values
# point to existing NPY files.

if unresolved_cxr:

    candidate_columns = [

        "npy_path",
        "npy_file",
        "input_npy",
        "input_path",
        "coanet_npy",
        "physical_npy",
        "output_npy",
        "file_path",
        "path",

    ]

    usable_column = None

    for column in candidate_columns:

        if column not in cxr_manifest.columns:

            continue

        sample = (
            cxr_manifest[
                column
            ]
            .dropna()
            .astype(str)
            .head(20)
            .tolist()
        )

        if not sample:

            continue

        existing = 0

        for value in sample:

            candidate = Path(
                value
            )

            if candidate.exists():

                existing += 1

            else:

                candidate2 = (
                    CXR_NPY_ROOT
                    / Path(value).name
                )

                if candidate2.exists():

                    existing += 1

        if existing > 0:

            usable_column = column
            break


    if usable_column is not None:

        for _, row in cxr_manifest.iterrows():

            condition_id = (
                row[
                    "_condition_id_norm"
                ]
            )

            raw_value = row[
                usable_column
            ]

            if pd.isna(raw_value):

                continue

            candidate = Path(
                str(raw_value)
            )

            if candidate.exists():

                resolved = candidate

            else:

                resolved = (
                    CXR_NPY_ROOT
                    / candidate.name
                )

            if resolved.exists():

                cxr_paths[
                    condition_id
                ] = resolved


    unresolved_cxr = [
        condition_id
        for condition_id in cxr_manifest[
            "_condition_id_norm"
        ]
        if condition_id not in cxr_paths
    ]


if unresolved_cxr:

    raise RuntimeError(
        "Could not resolve all 1976 CXR NPY files "
        "without guessing.\n"
        f"Unresolved conditions: "
        f"{len(unresolved_cxr)}\n"
        f"First examples: "
        f"{unresolved_cxr[:10]}\n"
        f"Manifest columns: "
        f"{list(cxr_manifest.columns)}"
    )


if len(cxr_paths) != 1976:

    raise RuntimeError(
        "CXR path resolution did not produce exactly 1976 files."
    )

print(
    "Resolved CXR NPYs:",
    len(cxr_paths)
)

print(
    "CXR NPY resolution: PASS"
)


# =============================================================================
# 14. CXR DATASET
# =============================================================================

class CXRFeatureDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        path_map
    ):

        self.df = (
            dataframe
            .reset_index(
                drop=True
            )
        )

        self.path_map = path_map

    def __len__(
        self
    ):

        return len(
            self.df
        )

    def __getitem__(
        self,
        index
    ):

        row = self.df.iloc[
            index
        ]

        condition_id = (
            row[
                "_condition_id_norm"
            ]
        )

        npy_path = self.path_map[
            condition_id
        ]

        array = np.load(
            npy_path,
            allow_pickle=False
        )

        array = np.asarray(
            array,
            dtype=np.float32
        )

        # Expected stored representation:
        # HWC, float32, 224x224x3, range [-1, 1].

        if array.shape != (
            224,
            224,
            3
        ):

            raise RuntimeError(
                f"Unexpected CXR shape for "
                f"{condition_id}: "
                f"{array.shape}"
            )

        if not np.isfinite(
            array
        ).all():

            raise RuntimeError(
                f"Non-finite CXR values: "
                f"{condition_id}"
            )

        min_value = float(
            array.min()
        )

        max_value = float(
            array.max()
        )

        if (
            min_value < -1.0001
            or
            max_value > 1.0001
        ):

            raise RuntimeError(
                f"CXR range outside [-1,1] "
                f"for {condition_id}: "
                f"{min_value}, {max_value}"
            )

        # [-1,1] -> [0,1]
        array = (
            array + 1.0
        ) / 2.0

        # HWC -> CHW
        tensor = torch.from_numpy(
            array.transpose(
                2,
                0,
                1
            ).copy()
        )

        # Exact ImageNet normalization used by Run-2.
        mean = torch.tensor(
            [
                0.485,
                0.456,
                0.406
            ],
            dtype=torch.float32
        ).view(
            3,
            1,
            1
        )

        std = torch.tensor(
            [
                0.229,
                0.224,
                0.225
            ],
            dtype=torch.float32
        ).view(
            3,
            1,
            1
        )

        tensor = (
            tensor - mean
        ) / std

        return {

            "image":
                tensor,

            "condition_id":
                condition_id,

            "target":
                int(
                    row[
                        "target_binary"
                    ]
                ),

            "split":
                row[
                    "split"
                ],
        }


# =============================================================================
# 15. CXR DATALOADER
# =============================================================================

cxr_dataset = CXRFeatureDataset(
    split_df,
    cxr_paths
)

cxr_loader = DataLoader(
    cxr_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)


# =============================================================================
# 16. LOAD FROZEN CoAtNet
# =============================================================================

print("\nCXR ENCODER")
print("-" * 80)

print(
    "Architecture:",
    "coatnet_0_rw_224"
)

print(
    "Checkpoint:",
    CXR_CHECKPOINT
)

print(
    "Frozen checkpoint epoch:",
    7
)

cxr_model = timm.create_model(
    "coatnet_0_rw_224",
    pretrained=True,
    num_classes=2
)


cxr_checkpoint = torch.load(
    CXR_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)


# Robustly handle the checkpoint structure.
if (
    isinstance(
        cxr_checkpoint,
        dict
    )
    and
    "model_state_dict" in cxr_checkpoint
):

    cxr_state_dict = (
        cxr_checkpoint[
            "model_state_dict"
        ]
    )

elif (
    isinstance(
        cxr_checkpoint,
        dict
    )
    and
    "state_dict" in cxr_checkpoint
):

    cxr_state_dict = (
        cxr_checkpoint[
            "state_dict"
        ]
    )

else:

    cxr_state_dict = cxr_checkpoint


# Remove DataParallel prefix if present.
clean_cxr_state_dict = {}

for key, value in cxr_state_dict.items():

    clean_key = key

    if clean_key.startswith(
        "module."
    ):

        clean_key = clean_key[
            len("module.") :
        ]

    clean_cxr_state_dict[
        clean_key
    ] = value


load_result = (
    cxr_model.load_state_dict(
        clean_cxr_state_dict,
        strict=True
    )
)

print(
    "Checkpoint loaded: PASS"
)

print(
    "Missing keys:",
    len(
        load_result.missing_keys
    )
)

print(
    "Unexpected keys:",
    len(
        load_result.unexpected_keys
    )
)

cxr_model.eval()


for parameter in (
    cxr_model.parameters()
):

    parameter.requires_grad = False


# =============================================================================
# 17. CXR EMBEDDING EXTRACTION
# =============================================================================

print("\nCXR EMBEDDING EXTRACTION")
print("-" * 80)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

cxr_model = cxr_model.to(
    device
)

cxr_embeddings = []
cxr_ids = []
cxr_targets = []
cxr_splits = []


with torch.no_grad():

    for batch in cxr_loader:

        images = batch[
            "image"
        ].to(device)

        # timm CoAtNet:
        # forward_features -> backbone feature map
        # forward_head(pre_logits=True) -> 768-D representation.

        features = (
            cxr_model.forward_features(
                images
            )
        )

        embeddings = (
            cxr_model.forward_head(
                features,
                pre_logits=True
            )
        )

        if embeddings.ndim != 2:

            raise RuntimeError(
                "CXR embedding output is not 2-D."
            )

        if embeddings.shape[1] != 768:

            raise RuntimeError(
                f"Expected 768-D CXR embedding, "
                f"got {embeddings.shape}"
            )

        if not torch.isfinite(
            embeddings
        ).all():

            raise RuntimeError(
                "Non-finite CXR embeddings detected."
            )

        cxr_embeddings.append(
            embeddings.cpu().numpy()
        )

        cxr_ids.extend(
            list(
                batch[
                    "condition_id"
                ]
            )
        )

        cxr_targets.extend(
            batch[
                "target"
            ].numpy().tolist()
        )

        cxr_splits.extend(
            list(
                batch[
                    "split"
                ]
            )
        )


cxr_embeddings = np.concatenate(
    cxr_embeddings,
    axis=0
)


# =============================================================================
# 18. CXR EMBEDDING GATE
# =============================================================================

print(
    "CXR embedding shape:",
    cxr_embeddings.shape
)

if cxr_embeddings.shape != (
    1976,
    768
):

    raise RuntimeError(
        "CXR embedding shape must be (1976, 768)."
    )

if len(
    set(cxr_ids)
) != 1976:

    raise RuntimeError(
        "CXR embedding condition IDs are not unique."
    )

print(
    "CXR embeddings: PASS"
)


# =============================================================================
# 19. LOAD GENOMIC DATA
# =============================================================================

print("\nGENOMIC ENCODER")
print("-" * 80)

genomic_df = pd.read_csv(
    GENOMIC_SEQUENCE_FILE,
    low_memory=False
)

with open(
    GENOMIC_VOCAB_FILE,
    "r",
    encoding="utf-8"
) as f:

    vocab_data = json.load(f)

with open(
    GENOMIC_TRAINING_SUMMARY,
    "r",
    encoding="utf-8"
) as f:

    genomic_training_summary = json.load(f)


if len(genomic_df) != 1976:

    raise RuntimeError(
        "Genomic dataset must contain 1976 conditions."
    )

genomic_df[
    "_condition_id_norm"
] = normalize_condition_id(
    genomic_df[
        "condition_id"
    ]
)


if (
    genomic_df[
        "_condition_id_norm"
    ].nunique()
    != 1976
):

    raise RuntimeError(
        "Genomic condition IDs are not unique."
    )


genomic_best_epoch = int(
    genomic_training_summary[
        "best_epoch"
    ]
)

if genomic_best_epoch != 11:

    raise RuntimeError(
        f"Expected genomic best epoch 11, "
        f"found {genomic_best_epoch}."
    )

print(
    "Frozen checkpoint epoch:",
    genomic_best_epoch
)

print(
    "Genomic encoder checkpoint: PASS"
)


# =============================================================================
# 20. PARSE GENOMIC SEQUENCES
# =============================================================================

def parse_array(value):

    value = str(
        value
    ).strip()

    value = (
        value
        .replace("[", "")
        .replace("]", "")
    )

    if not value:

        return []

    return [
        int(x.strip())
        for x in value.split(",")
        if x.strip()
    ]


genomic_df[
    "input_ids_parsed"
] = genomic_df[
    "input_ids"
].apply(
    parse_array
)

genomic_df[
    "attention_mask_parsed"
] = genomic_df[
    "attention_mask"
].apply(
    parse_array
)


MAX_SEQ_LEN = int(
    vocab_data[
        "max_seq_len"
    ]
)

VOCAB_SIZE = int(
    vocab_data[
        "vocab_size"
    ]
)

PAD_ID = int(
    vocab_data[
        "pad_id"
    ]
)


# =============================================================================
# 21. GENOMIC DATASET
# =============================================================================

class GenomicEmbeddingDataset(
    Dataset
):

    def __init__(
        self,
        dataframe
    ):

        self.df = (
            dataframe
            .reset_index(
                drop=True
            )
        )

        self.input_ids = np.asarray(
            self.df[
                "input_ids_parsed"
            ].tolist(),
            dtype=np.int64
        )

        self.attention_masks = np.asarray(
            self.df[
                "attention_mask_parsed"
            ].tolist(),
            dtype=np.int64
        )

        self.targets = (
            self.df[
                "target_binary"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        self.condition_ids = (
            self.df[
                "_condition_id_norm"
            ]
            .astype(str)
            .to_numpy()
        )

        self.splits = (
            self.df[
                "split"
            ]
            .astype(str)
            .to_numpy()
        )

    def __len__(
        self
    ):

        return len(
            self.targets
        )

    def __getitem__(
        self,
        index
    ):

        return {

            "input_ids":
                torch.tensor(
                    self.input_ids[index],
                    dtype=torch.long
                ),

            "attention_mask":
                torch.tensor(
                    self.attention_masks[index],
                    dtype=torch.long
                ),

            "target":
                torch.tensor(
                    self.targets[index],
                    dtype=torch.long
                ),

            "condition_id":
                self.condition_ids[index],

            "split":
                self.splits[index],

        }


genomic_dataset = GenomicEmbeddingDataset(
    genomic_df
)

genomic_loader = DataLoader(
    genomic_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)


# =============================================================================
# 22. GENOMIC TRANSFORMER
# =============================================================================

class GenomicTransformer(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        embed_dim=128,
        num_heads=4,
        num_layers=2,
        ff_dim=256,
        dropout=0.20,
        max_seq_len=74,
        num_classes=2,
        pad_id=0
    ):

        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=pad_id
        )

        self.position_embedding = nn.Embedding(
            max_seq_len,
            embed_dim
        )

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=embed_dim,
                nhead=num_heads,
                dim_feedforward=ff_dim,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=False
            )
        )

        self.encoder = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=num_layers
            )
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(
                embed_dim
            ),
            nn.Linear(
                embed_dim,
                num_classes
            )
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        return_embedding=False
    ):

        batch_size, seq_len = (
            input_ids.shape
        )

        positions = torch.arange(
            seq_len,
            device=input_ids.device
        )

        positions = (
            positions
            .unsqueeze(0)
            .expand(
                batch_size,
                seq_len
            )
        )

        x = (
            self.token_embedding(
                input_ids
            )
            +
            self.position_embedding(
                positions
            )
        )

        x = self.dropout(
            x
        )

        padding_mask = (
            attention_mask == 0
        )

        x = self.encoder(
            x,
            src_key_padding_mask=padding_mask
        )

        cls_embedding = x[:, 0, :]

        cls_embedding = self.dropout(
            cls_embedding
        )

        logits = self.classifier(
            cls_embedding
        )

        if return_embedding:

            return (
                logits,
                cls_embedding
            )

        return logits


# =============================================================================
# 23. LOAD FROZEN GENOMIC CHECKPOINT
# =============================================================================

genomic_model = GenomicTransformer(
    vocab_size=VOCAB_SIZE,
    embed_dim=128,
    num_heads=4,
    num_layers=2,
    ff_dim=256,
    dropout=0.20,
    max_seq_len=MAX_SEQ_LEN,
    num_classes=2,
    pad_id=PAD_ID
).to(device)


genomic_checkpoint = torch.load(
    GENOMIC_CHECKPOINT,
    map_location=device,
    weights_only=False
)


genomic_state_dict = (
    genomic_checkpoint[
        "model_state_dict"
    ]
)

clean_genomic_state_dict = {}

for key, value in genomic_state_dict.items():

    clean_key = key

    if clean_key.startswith(
        "module."
    ):

        clean_key = clean_key[
            len("module.") :
        ]

    clean_genomic_state_dict[
        clean_key
    ] = value


genomic_model.load_state_dict(
    clean_genomic_state_dict,
    strict=True
)

genomic_model.eval()

for parameter in (
    genomic_model.parameters()
):

    parameter.requires_grad = False


print(
    "Genomic checkpoint loaded: PASS"
)


# =============================================================================
# 24. GENOMIC EMBEDDING EXTRACTION
# =============================================================================

print("\nGENOMIC EMBEDDING EXTRACTION")
print("-" * 80)

genomic_embeddings = []
genomic_ids = []
genomic_targets = []
genomic_splits = []


with torch.no_grad():

    for batch in genomic_loader:

        input_ids = batch[
            "input_ids"
        ].to(device)

        attention_mask = batch[
            "attention_mask"
        ].to(device)

        logits, embeddings = (
            genomic_model(
                input_ids,
                attention_mask,
                return_embedding=True
            )
        )

        if embeddings.ndim != 2:

            raise RuntimeError(
                "Genomic embedding output is not 2-D."
            )

        if embeddings.shape[1] != 128:

            raise RuntimeError(
                f"Expected 128-D genomic embedding, "
                f"got {embeddings.shape}"
            )

        if not torch.isfinite(
            embeddings
        ).all():

            raise RuntimeError(
                "Non-finite genomic embeddings detected."
            )

        genomic_embeddings.append(
            embeddings.cpu().numpy()
        )

        genomic_ids.extend(
            list(
                batch[
                    "condition_id"
                ]
            )
        )

        genomic_targets.extend(
            batch[
                "target"
            ].numpy().tolist()
        )

        genomic_splits.extend(
            list(
                batch[
                    "split"
                ]
            )
        )


genomic_embeddings = np.concatenate(
    genomic_embeddings,
    axis=0
)


# =============================================================================
# 25. GENOMIC EMBEDDING GATE
# =============================================================================

print(
    "Genomic embedding shape:",
    genomic_embeddings.shape
)

if genomic_embeddings.shape != (
    1976,
    128
):

    raise RuntimeError(
        "Genomic embedding shape must be (1976, 128)."
    )

if len(
    set(genomic_ids)
) != 1976:

    raise RuntimeError(
        "Genomic embedding condition IDs are not unique."
    )

print(
    "Genomic embeddings: PASS"
)


# =============================================================================
# 26. DIRECT CXR ↔ GENOMIC ID ALIGNMENT
# =============================================================================

cxr_id_set = set(
    cxr_ids
)

genomic_id_set = set(
    genomic_ids
)

if cxr_id_set != genomic_id_set:

    missing_from_cxr = (
        genomic_id_set
        -
        cxr_id_set
    )

    missing_from_genomic = (
        cxr_id_set
        -
        genomic_id_set
    )

    raise RuntimeError(
        "CXR/genomic embedding ID mismatch.\n"
        f"Missing from CXR: "
        f"{len(missing_from_cxr)}\n"
        f"Missing from genomic: "
        f"{len(missing_from_genomic)}"
    )

print("\nEMBEDDING ID ALIGNMENT")
print("-" * 80)

print(
    "CXR IDs:",
    len(cxr_id_set)
)

print(
    "Genomic IDs:",
    len(genomic_id_set)
)

print(
    "Common IDs:",
    len(
        cxr_id_set & genomic_id_set
    )
)

print(
    "CXR ↔ genomic embedding alignment: PASS"
)


# =============================================================================
# 27. CREATE CONDITION-ALIGNED EMBEDDING TABLE
# =============================================================================

cxr_embedding_df = pd.DataFrame(
    cxr_embeddings,
    columns=[
        f"cxr_emb_{i:03d}"
        for i in range(768)
    ]
)

cxr_embedding_df[
    "condition_id"
] = cxr_ids

cxr_embedding_df[
    "target_binary"
] = cxr_targets

cxr_embedding_df[
    "split"
] = cxr_splits


genomic_embedding_df = pd.DataFrame(
    genomic_embeddings,
    columns=[
        f"genomic_emb_{i:03d}"
        for i in range(128)
    ]
)

genomic_embedding_df[
    "condition_id"
] = genomic_ids

genomic_embedding_df[
    "target_binary"
] = genomic_targets

genomic_embedding_df[
    "split"
] = genomic_splits


# Normalize ID column before merge.

cxr_embedding_df[
    "_condition_id_norm"
] = normalize_condition_id(
    cxr_embedding_df[
        "condition_id"
    ]
)

genomic_embedding_df[
    "_condition_id_norm"
] = normalize_condition_id(
    genomic_embedding_df[
        "condition_id"
    ]
)


# =============================================================================
# 28. MERGE EMBEDDINGS
# =============================================================================

merged_embeddings = pd.merge(
    cxr_embedding_df,
    genomic_embedding_df,
    on="_condition_id_norm",
    how="inner",
    suffixes=(
        "_cxr",
        "_genomic"
    ),
    validate="one_to_one"
)


if len(
    merged_embeddings
) != 1976:

    raise RuntimeError(
        "Multimodal embedding merge must contain "
        "exactly 1976 conditions."
    )


# =============================================================================
# 29. TARGET / SPLIT CONSISTENCY AFTER MERGE
# =============================================================================

if not np.array_equal(
    merged_embeddings[
        "target_binary_cxr"
    ].to_numpy(),
    merged_embeddings[
        "target_binary_genomic"
    ].to_numpy()
):

    raise RuntimeError(
        "Target mismatch after embedding merge."
    )


if not (
    merged_embeddings[
        "split_cxr"
    ]
    ==
    merged_embeddings[
        "split_genomic"
    ]
).all():

    raise RuntimeError(
        "Split mismatch after embedding merge."
    )


merged_embeddings[
    "target_binary"
] = merged_embeddings[
    "target_binary_cxr"
].astype(int)

merged_embeddings[
    "split"
] = merged_embeddings[
    "split_cxr"
].astype(str)


# =============================================================================
# 30. REORDER BY AUTHORITATIVE FROZEN SPLIT
# =============================================================================

authoritative_order = (
    split_df[
        [
            "_condition_id_norm",
            "condition_id",
            "target_binary",
            "split"
        ]
    ]
    .copy()
)

authoritative_order[
    "_row_order"
] = np.arange(
    len(authoritative_order)
)

merged_embeddings = (
    authoritative_order[
        [
            "_condition_id_norm"
        ]
    ]
    .merge(
        merged_embeddings,
        on="_condition_id_norm",
        how="left",
        validate="one_to_one"
    )
    .sort_values(
        "_row_order"
        if "_row_order" in merged_embeddings.columns
        else "_condition_id_norm"
    )
)


# Rebuild in a deterministic way using the authoritative order.
merged_lookup = {
    row["_condition_id_norm"]: row
    for _, row in merged_embeddings.iterrows()
}

ordered_rows = []

for _, split_row in split_df.iterrows():

    cid = split_row[
        "_condition_id_norm"
    ]

    if cid not in merged_lookup:

        raise RuntimeError(
            f"Missing merged embedding: {cid}"
        )

    ordered_rows.append(
        merged_lookup[cid]
    )

merged_embeddings = pd.DataFrame(
    ordered_rows
).reset_index(
    drop=True
)


# =============================================================================
# 31. EXTRACT FINAL NUMPY MATRICES
# =============================================================================

cxr_columns = [
    f"cxr_emb_{i:03d}"
    for i in range(768)
]

genomic_columns = [
    f"genomic_emb_{i:03d}"
    for i in range(128)
]


final_cxr_embeddings = (
    merged_embeddings[
        cxr_columns
    ]
    .to_numpy(
        dtype=np.float32
    )
)

final_genomic_embeddings = (
    merged_embeddings[
        genomic_columns
    ]
    .to_numpy(
        dtype=np.float32
    )
)

final_targets = (
    merged_embeddings[
        "target_binary"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

final_splits = (
    merged_embeddings[
        "split"
    ]
    .astype(str)
    .to_numpy()
)

final_condition_ids = (
    split_df[
        "condition_id"
    ]
    .astype(str)
    .to_numpy()
)


# =============================================================================
# 32. FINAL EMBEDDING MATRIX GATES
# =============================================================================

print("\nFINAL EMBEDDING MATRICES")
print("-" * 80)

print(
    "CXR:",
    final_cxr_embeddings.shape
)

print(
    "Genomic:",
    final_genomic_embeddings.shape
)

print(
    "Targets:",
    final_targets.shape
)

print(
    "Condition IDs:",
    final_condition_ids.shape
)

if final_cxr_embeddings.shape != (
    1976,
    768
):

    raise RuntimeError(
        "Final CXR embedding matrix shape incorrect."
    )

if final_genomic_embeddings.shape != (
    1976,
    128
):

    raise RuntimeError(
        "Final genomic embedding matrix shape incorrect."
    )

if final_targets.shape != (
    1976,
):

    raise RuntimeError(
        "Final target vector shape incorrect."
    )

if final_condition_ids.shape != (
    1976,
):

    raise RuntimeError(
        "Final condition ID vector shape incorrect."
    )

if not np.isfinite(
    final_cxr_embeddings
).all():

    raise RuntimeError(
        "Non-finite CXR embeddings."
    )

if not np.isfinite(
    final_genomic_embeddings
).all():

    raise RuntimeError(
        "Non-finite genomic embeddings."
    )

print(
    "Embedding matrix gates: PASS"
)


# =============================================================================
# 33. SPLIT DISTRIBUTION
# =============================================================================

print("\nFINAL SPLIT DISTRIBUTION")
print("-" * 80)

for split_name in [
    "train",
    "validation",
    "test"
]:

    mask = (
        final_splits
        ==
        split_name
    )

    ds_count = int(
        (
            final_targets[mask]
            ==
            0
        ).sum()
    )

    dr_count = int(
        (
            final_targets[mask]
            ==
            1
        ).sum()
    )

    print(
        f"{split_name:<12}: "
        f"{int(mask.sum())} | "
        f"DS={ds_count} | "
        f"DR={dr_count}"
    )


# =============================================================================
# 34. VERIFY AGAINST EXISTING GENOMIC TEST EMBEDDINGS
# =============================================================================

print("\nGENOMIC TEST EMBEDDING CONSISTENCY CHECK")
print("-" * 80)

previous_test_embeddings = np.load(
    GENOMIC_PREVIOUS_TEST_EMBEDDINGS
)

previous_test_ids = np.load(
    GENOMIC_PREVIOUS_TEST_IDS,
    allow_pickle=True
).astype(str
)

test_mask = (
    final_splits
    ==
    "test"
)

current_test_ids = (
    final_condition_ids[
        test_mask
    ]
)

current_test_embeddings = (
    final_genomic_embeddings[
        test_mask
    ]
)


previous_test_id_set = set(
    normalize_condition_id(
        pd.Series(
            previous_test_ids
        )
    )
)

current_test_id_set = set(
    normalize_condition_id(
        pd.Series(
            current_test_ids
        )
    )
)

if (
    previous_test_id_set
    !=
    current_test_id_set
):

    raise RuntimeError(
        "Previously saved genomic test IDs do not "
        "match the current frozen test IDs."
    )


# Align previous embeddings to current test order.

previous_test_map = {

    str(
        condition_id
    ).strip().lower():
        embedding

    for condition_id, embedding
    in zip(
        previous_test_ids,
        previous_test_embeddings
    )

}

aligned_previous_test_embeddings = np.vstack(
    [
        previous_test_map[
            str(
                condition_id
            ).strip().lower()
        ]
        for condition_id in current_test_ids
    ]
)


max_test_embedding_difference = float(
    np.max(
        np.abs(
            current_test_embeddings
            -
            aligned_previous_test_embeddings
        )
    )
)


print(
    "Previous test embeddings:",
    previous_test_embeddings.shape
)

print(
    "Current test embeddings:",
    current_test_embeddings.shape
)

print(
    "Maximum absolute difference:",
    f"{max_test_embedding_difference:.10f}"
)

if max_test_embedding_difference > 1e-6:

    raise RuntimeError(
        "Current frozen genomic embeddings differ from "
        "the previously saved Cell 8 test embeddings."
    )

print(
    "Genomic test embedding consistency: PASS"
)


# =============================================================================
# 35. SAVE EMBEDDINGS
# =============================================================================

CXR_EMBEDDINGS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_CXR_Embeddings_768D.npy"
)

GENOMIC_EMBEDDINGS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_Genomic_Embeddings_128D.npy"
)

TARGETS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_Targets.npy"
)

CONDITION_IDS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_Condition_IDs.npy"
)

SPLITS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_Splits.npy"
)


np.save(
    CXR_EMBEDDINGS_FILE,
    final_cxr_embeddings
)

np.save(
    GENOMIC_EMBEDDINGS_FILE,
    final_genomic_embeddings
)

np.save(
    TARGETS_FILE,
    final_targets
)

np.save(
    CONDITION_IDS_FILE,
    final_condition_ids
)

np.save(
    SPLITS_FILE,
    final_splits
)


# =============================================================================
# 36. SAVE COMBINED METADATA
# =============================================================================

metadata_df = pd.DataFrame(
    {
        "condition_id":
            final_condition_ids,

        "target_binary":
            final_targets,

        "split":
            final_splits,

    }
)

METADATA_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_Embedding_Metadata.csv"
)

metadata_df.to_csv(
    METADATA_FILE,
    index=False
)


# =============================================================================
# 37. SAVE CELL SUMMARY
# =============================================================================

cell2_summary = {

    "status":
        "PASS",

    "full_conditions":
        1976,

    "train_conditions":
        1185,

    "validation_conditions":
        395,

    "test_conditions":
        396,

    "cxr_encoder":
        "coatnet_0_rw_224",

    "cxr_checkpoint_epoch":
        7,

    "cxr_embedding_dimension":
        768,

    "genomic_checkpoint_epoch":
        11,

    "genomic_embedding_dimension":
        128,

    "cxr_embedding_shape":
        [
            1976,
            768
        ],

    "genomic_embedding_shape":
        [
            1976,
            128
        ],

    "target_alignment":
        "PASS",

    "condition_alignment":
        "PASS",

    "split_alignment":
        "PASS",

    "test_embedding_consistency":
        "PASS",

    "max_test_genomic_embedding_difference":
        max_test_embedding_difference,

    "training_performed":
        False,

    "fine_tuning_performed":
        False,

    "test_used_for_training":
        False,

    "test_used_for_checkpoint_selection":
        False,

    "test_used_for_threshold_tuning":
        False,

    "cxr_embeddings_file":
        str(
            CXR_EMBEDDINGS_FILE
        ),

    "genomic_embeddings_file":
        str(
            GENOMIC_EMBEDDINGS_FILE
        ),

    "metadata_file":
        str(
            METADATA_FILE
        )

}


SUMMARY_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_Frozen_Embedding_Extraction_Summary.json"
)

with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        cell2_summary,
        f,
        indent=4
    )


# =============================================================================
# 38. FINAL STATUS
# =============================================================================

print("\n")
print("=" * 80)
print("CELL 2 STATUS: PASS")
print("=" * 80)

print(
    "CXR encoder checkpoint epoch:",
    7
)

print(
    "Genomic encoder checkpoint epoch:",
    11
)

print(
    "CXR embeddings:",
    final_cxr_embeddings.shape
)

print(
    "Genomic embeddings:",
    final_genomic_embeddings.shape
)

print(
    "Conditions:",
    1976
)

print(
    "Training performed:",
    "NO"
)

print(
    "Fine-tuning performed:",
    "NO"
)

print(
    "Test used for training:",
    "NO"
)

print(
    "Test used for checkpoint selection:",
    "NO"
)

print(
    "\nCXR embeddings:"
)

print(
    CXR_EMBEDDINGS_FILE
)

print(
    "\nGenomic embeddings:"
)

print(
    GENOMIC_EMBEDDINGS_FILE
)

print(
    "\nMetadata:"
)

print(
    METADATA_FILE
)

print(
    "\nSummary:"
)

print(
    SUMMARY_FILE
)

print(
    "\nSTOP HERE."
)

print(
    "Do not train the multimodal fusion model in this cell."
)

NOTEBOOK 15 — CELL 2
FINAL FROZEN CXR + GENOMIC EMBEDDING EXTRACTION

PATH GATE
--------------------------------------------------------------------------------
Frozen split                        : PASS
CXR manifest                        : PASS
CXR NPY root                        : PASS
CXR checkpoint                      : PASS
Genomic sequence dataset            : PASS
Genomic vocabulary                  : PASS
Genomic checkpoint                  : PASS
Genomic training summary            : PASS
Multimodal master                   : PASS
Cell 1 summary                      : PASS
Previous genomic test embeddings    : PASS
Previous genomic test IDs           : PASS

FROZEN SPLIT
--------------------------------------------------------------------------------
Conditions: 1976
Train: 1185
Validation: 395
Test: 396
Hash: 74b0a5e4c757ba047f75d9b2dd5fcc64f2f6a3a9a63dc22b69dd71a52554911f
Frozen split: PASS

CXR NPY RESOLUTION
---------------------------------------------------------------

Checkpoint loaded: PASS
Missing keys: 0
Unexpected keys: 0

CXR EMBEDDING EXTRACTION
--------------------------------------------------------------------------------
CXR embedding shape: (1976, 768)
CXR embeddings: PASS

GENOMIC ENCODER
--------------------------------------------------------------------------------
Frozen checkpoint epoch: 11
Genomic encoder checkpoint: PASS
Genomic checkpoint loaded: PASS

GENOMIC EMBEDDING EXTRACTION
--------------------------------------------------------------------------------


C:\Users\Gobika\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\transformer.py:529: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:181.)
  output = torch._nested_tensor_from_mask(


Genomic embedding shape: (1976, 128)
Genomic embeddings: PASS

EMBEDDING ID ALIGNMENT
--------------------------------------------------------------------------------
CXR IDs: 1976
Genomic IDs: 1976
Common IDs: 1976
CXR ↔ genomic embedding alignment: PASS

FINAL EMBEDDING MATRICES
--------------------------------------------------------------------------------
CXR: (1976, 768)
Genomic: (1976, 128)
Targets: (1976,)
Condition IDs: (1976,)
Embedding matrix gates: PASS

FINAL SPLIT DISTRIBUTION
--------------------------------------------------------------------------------
train       : 1185 | DS=422 | DR=763
validation  : 395 | DS=141 | DR=254
test        : 396 | DS=141 | DR=255

GENOMIC TEST EMBEDDING CONSISTENCY CHECK
--------------------------------------------------------------------------------
Previous test embeddings: (396, 128)
Current test embeddings: (396, 128)
Maximum absolute difference: 0.0000000000
Genomic test embedding consistency: PASS


CELL 2 STATUS: PASS
CXR encoder c

In [3]:
# =============================================================================
# NOTEBOOK 15 — CELL 3
# FINAL MULTIMODAL CXR + GENOMIC FUSION ARCHITECTURE AUDIT
#
# CXR representation     : 768-D
# Genomic representation : 128-D
#
# TRAINING: NO
# TEST EVALUATION: NO
# CHECKPOINT SELECTION: NO
# THRESHOLD TUNING: NO
# =============================================================================

from pathlib import Path
import json
import hashlib
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn


print("=" * 80)
print("NOTEBOOK 15 — CELL 3")
print("FINAL MULTIMODAL FUSION ARCHITECTURE AUDIT")
print("=" * 80)


# =============================================================================
# 1. REPRODUCIBILITY
# =============================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# =============================================================================
# 2. PROJECT PATHS
# =============================================================================

PROJECT_ROOT = Path(
    r"C:\TBP\Metadata"
)

FINAL_ROOT = (
    PROJECT_ROOT
    / "Step_3B_8_CXR_CoAtNet_Preparation"
    / "QC"
    / "Clean_PSPNet_CXR_Cohort"
    / "FINAL_FROZEN_CLEAN_COHORT"
)

MULTIMODAL_ROOT = (
    FINAL_ROOT
    / "Multimodal_15"
)

CELL1_ROOT = (
    MULTIMODAL_ROOT
    / "Cell1_Alignment_Audit"
)

CELL2_ROOT = (
    MULTIMODAL_ROOT
    / "Cell2_Frozen_Embedding_Extraction"
)

CELL3_ROOT = (
    MULTIMODAL_ROOT
    / "Cell3_Fusion_Architecture"
)

CELL3_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 3. INPUT FILES
# =============================================================================

MULTIMODAL_MASTER = (
    CELL1_ROOT
    / "Notebook15_Cell1_FINAL_1976_Multimodal_Master.csv"
)

CELL1_SUMMARY = (
    CELL1_ROOT
    / "Notebook15_Cell1_Multimodal_Alignment_Audit_Summary.json"
)

CXR_EMBEDDINGS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_CXR_Embeddings_768D.npy"
)

GENOMIC_EMBEDDINGS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_Genomic_Embeddings_128D.npy"
)

TARGETS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_Targets.npy"
)

CONDITION_IDS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_Condition_IDs.npy"
)

SPLITS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_Splits.npy"
)

CELL2_SUMMARY = (
    CELL2_ROOT
    / "Notebook15_Cell2_Frozen_Embedding_Extraction_Summary.json"
)


# =============================================================================
# 4. PATH GATE
# =============================================================================

print("\nPATH GATE")
print("-" * 80)

required_paths = {

    "Multimodal master":
        MULTIMODAL_MASTER,

    "Cell 1 summary":
        CELL1_SUMMARY,

    "CXR embeddings":
        CXR_EMBEDDINGS_FILE,

    "Genomic embeddings":
        GENOMIC_EMBEDDINGS_FILE,

    "Targets":
        TARGETS_FILE,

    "Condition IDs":
        CONDITION_IDS_FILE,

    "Splits":
        SPLITS_FILE,

    "Cell 2 summary":
        CELL2_SUMMARY,

}

for name, path in required_paths.items():

    if not path.exists():

        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

    print(
        f"{name:<30}: PASS"
    )


# =============================================================================
# 5. LOAD PREVIOUS CELL SUMMARIES
# =============================================================================

with open(
    CELL1_SUMMARY,
    "r",
    encoding="utf-8"
) as f:

    cell1_summary = json.load(f)


with open(
    CELL2_SUMMARY,
    "r",
    encoding="utf-8"
) as f:

    cell2_summary = json.load(f)


if cell1_summary.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Notebook 15 Cell 1 did not PASS."
    )


if cell2_summary.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Notebook 15 Cell 2 did not PASS."
    )


if cell2_summary.get(
    "training_performed"
) is not False:

    raise RuntimeError(
        "Cell 2 reports unexpected training."
    )


if cell2_summary.get(
    "fine_tuning_performed"
) is not False:

    raise RuntimeError(
        "Cell 2 reports unexpected fine-tuning."
    )


# =============================================================================
# 6. LOAD MULTIMODAL DATA
# =============================================================================

multimodal_master = pd.read_csv(
    MULTIMODAL_MASTER,
    low_memory=False
)

cxr_embeddings = np.load(
    CXR_EMBEDDINGS_FILE
)

genomic_embeddings = np.load(
    GENOMIC_EMBEDDINGS_FILE
)

targets = np.load(
    TARGETS_FILE
)

condition_ids = np.load(
    CONDITION_IDS_FILE,
    allow_pickle=True
).astype(str)

splits = np.load(
    SPLITS_FILE,
    allow_pickle=True
).astype(str)


# =============================================================================
# 7. DATASET SHAPE GATE
# =============================================================================

print("\nDATASET SHAPE GATE")
print("-" * 80)

print(
    "Multimodal master:",
    multimodal_master.shape
)

print(
    "CXR embeddings:",
    cxr_embeddings.shape
)

print(
    "Genomic embeddings:",
    genomic_embeddings.shape
)

print(
    "Targets:",
    targets.shape
)

print(
    "Condition IDs:",
    condition_ids.shape
)

print(
    "Splits:",
    splits.shape
)


if len(
    multimodal_master
) != 1976:

    raise RuntimeError(
        "Multimodal master must contain 1976 conditions."
    )


if cxr_embeddings.shape != (
    1976,
    768
):

    raise RuntimeError(
        "CXR embeddings must have shape (1976, 768)."
    )


if genomic_embeddings.shape != (
    1976,
    128
):

    raise RuntimeError(
        "Genomic embeddings must have shape (1976, 128)."
    )


if targets.shape != (
    1976,
):

    raise RuntimeError(
        "Targets must have shape (1976,)."
    )


if condition_ids.shape != (
    1976,
):

    raise RuntimeError(
        "Condition IDs must have shape (1976,)."
    )


if splits.shape != (
    1976,
):

    raise RuntimeError(
        "Splits must have shape (1976,)."
    )


print(
    "Dataset shape gate: PASS"
)


# =============================================================================
# 8. FINITE-VALUE AUDIT
# =============================================================================

print("\nFINITE-VALUE AUDIT")
print("-" * 80)

if not np.isfinite(
    cxr_embeddings
).all():

    raise RuntimeError(
        "Non-finite CXR embeddings detected."
    )


if not np.isfinite(
    genomic_embeddings
).all():

    raise RuntimeError(
        "Non-finite genomic embeddings detected."
    )


if not np.isfinite(
    targets
).all():

    raise RuntimeError(
        "Non-finite targets detected."
    )


print(
    "CXR embeddings: finite"
)

print(
    "Genomic embeddings: finite"
)

print(
    "Targets: finite"
)

print(
    "Finite-value audit: PASS"
)


# =============================================================================
# 9. CONDITION-ID UNIQUENESS
# =============================================================================

print("\nCONDITION-ID AUDIT")
print("-" * 80)

unique_condition_count = len(
    set(
        condition_ids
    )
)

print(
    "Unique condition IDs:",
    unique_condition_count
)

if unique_condition_count != 1976:

    raise RuntimeError(
        "Condition IDs are not unique."
    )

print(
    "Condition identity: PASS"
)


# =============================================================================
# 10. TARGET DISTRIBUTION
# =============================================================================

print("\nTARGET DISTRIBUTION")
print("-" * 80)

ds_count = int(
    (
        targets
        ==
        0
    ).sum()
)

dr_count = int(
    (
        targets
        ==
        1
    ).sum()
)

print(
    "DS:",
    ds_count
)

print(
    "DR:",
    dr_count
)

if ds_count != 704:

    raise RuntimeError(
        f"Expected 704 DS conditions, "
        f"found {ds_count}."
    )


if dr_count != 1272:

    raise RuntimeError(
        f"Expected 1272 DR conditions, "
        f"found {dr_count}."
    )


print(
    "Target distribution: PASS"
)


# =============================================================================
# 11. SPLIT DISTRIBUTION
# =============================================================================

print("\nSPLIT DISTRIBUTION")
print("-" * 80)

expected_split_sizes = {

    "train":
        1185,

    "validation":
        395,

    "test":
        396,

}

for split_name, expected_size in (
    expected_split_sizes.items()
):

    mask = (
        splits
        ==
        split_name
    )

    actual_size = int(
        mask.sum()
    )

    ds = int(
        (
            targets[mask]
            ==
            0
        ).sum()
    )

    dr = int(
        (
            targets[mask]
            ==
            1
        ).sum()
    )

    print(
        f"{split_name:<12}: "
        f"{actual_size} | "
        f"DS={ds} | DR={dr}"
    )

    if actual_size != expected_size:

        raise RuntimeError(
            f"{split_name} size mismatch."
        )


print(
    "Split distribution: PASS"
)


# =============================================================================
# 12. SPLIT OVERLAP AUDIT
# =============================================================================

train_ids = set(
    condition_ids[
        splits == "train"
    ]
)

validation_ids = set(
    condition_ids[
        splits == "validation"
    ]
)

test_ids = set(
    condition_ids[
        splits == "test"
    ]
)


print("\nSPLIT OVERLAP")
print("-" * 80)

print(
    "Train ∩ Validation:",
    len(
        train_ids
        &
        validation_ids
    )
)

print(
    "Train ∩ Test:",
    len(
        train_ids
        &
        test_ids
    )
)

print(
    "Validation ∩ Test:",
    len(
        validation_ids
        &
        test_ids
    )
)


if (
    train_ids
    &
    validation_ids
):

    raise RuntimeError(
        "Train/validation condition overlap."
    )


if (
    train_ids
    &
    test_ids
):

    raise RuntimeError(
        "Train/test condition overlap."
    )


if (
    validation_ids
    &
    test_ids
):

    raise RuntimeError(
        "Validation/test condition overlap."
    )


print(
    "Split overlap: PASS"
)


# =============================================================================
# 13. ARCHITECTURE CONFIGURATION
# =============================================================================

CXR_INPUT_DIM = 768
CXR_PROJECTION_DIM = 256

GENOMIC_INPUT_DIM = 128
GENOMIC_PROJECTION_DIM = 128

FUSION_INPUT_DIM = (
    CXR_PROJECTION_DIM
    +
    GENOMIC_PROJECTION_DIM
)

FUSION_HIDDEN_DIM = 128

NUM_CLASSES = 2

DROPOUT = 0.20


print("\nARCHITECTURE CONFIGURATION")
print("-" * 80)

print(
    "CXR input dimension:",
    CXR_INPUT_DIM
)

print(
    "CXR projection:",
    CXR_PROJECTION_DIM
)

print(
    "Genomic input dimension:",
    GENOMIC_INPUT_DIM
)

print(
    "Genomic projection:",
    GENOMIC_PROJECTION_DIM
)

print(
    "Fusion input dimension:",
    FUSION_INPUT_DIM
)

print(
    "Fusion hidden dimension:",
    FUSION_HIDDEN_DIM
)

print(
    "Output classes:",
    NUM_CLASSES
)

print(
    "Dropout:",
    DROPOUT
)


# =============================================================================
# 14. MULTIMODAL FUSION MODEL
# =============================================================================

class MultimodalFusionModel(
    nn.Module
):

    def __init__(
        self,
        cxr_input_dim=768,
        cxr_projection_dim=256,
        genomic_input_dim=128,
        genomic_projection_dim=128,
        fusion_hidden_dim=128,
        num_classes=2,
        dropout=0.20
    ):

        super().__init__()

        # ---------------------------------------------------------------------
        # CXR branch
        # ---------------------------------------------------------------------

        self.cxr_projection = nn.Sequential(

            nn.Linear(
                cxr_input_dim,
                cxr_projection_dim
            ),

            nn.LayerNorm(
                cxr_projection_dim
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            )

        )

        # ---------------------------------------------------------------------
        # Genomic branch
        # ---------------------------------------------------------------------

        self.genomic_projection = nn.Sequential(

            nn.Linear(
                genomic_input_dim,
                genomic_projection_dim
            ),

            nn.LayerNorm(
                genomic_projection_dim
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            )

        )

        # ---------------------------------------------------------------------
        # Fusion
        # ---------------------------------------------------------------------

        self.fusion = nn.Sequential(

            nn.Linear(
                cxr_projection_dim
                +
                genomic_projection_dim,
                fusion_hidden_dim
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                fusion_hidden_dim,
                num_classes
            )

        )

    def forward(
        self,
        cxr_embedding,
        genomic_embedding
    ):

        cxr_representation = (
            self.cxr_projection(
                cxr_embedding
            )
        )

        genomic_representation = (
            self.genomic_projection(
                genomic_embedding
            )
        )

        fused_representation = torch.cat(
            [
                cxr_representation,
                genomic_representation
            ],
            dim=1
        )

        logits = self.fusion(
            fused_representation
        )

        return {

            "logits":
                logits,

            "cxr_representation":
                cxr_representation,

            "genomic_representation":
                genomic_representation,

            "fused_representation":
                fused_representation,

        }


# =============================================================================
# 15. MODEL CREATION
# =============================================================================

print("\nMODEL CREATION")
print("-" * 80)

model = MultimodalFusionModel(
    cxr_input_dim=CXR_INPUT_DIM,
    cxr_projection_dim=CXR_PROJECTION_DIM,
    genomic_input_dim=GENOMIC_INPUT_DIM,
    genomic_projection_dim=GENOMIC_PROJECTION_DIM,
    fusion_hidden_dim=FUSION_HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT
)


# =============================================================================
# 16. PARAMETER AUDIT
# =============================================================================

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(
    "Total parameters:",
    total_parameters
)

print(
    "Trainable parameters:",
    trainable_parameters
)


# Expected:
#
# CXR projection:
# 768*256 + 256 = 196,864
# LayerNorm:
# 256 + 256 = 512
#
# Genomic projection:
# 128*128 + 128 = 16,512
# LayerNorm:
# 128 + 128 = 256
#
# Fusion:
# 384*128 + 128 = 49,280
# Output:
# 128*2 + 2 = 258
#
# Total = 263,682


EXPECTED_PARAMETER_COUNT = 263682

if total_parameters != EXPECTED_PARAMETER_COUNT:

    raise RuntimeError(
        "Unexpected multimodal parameter count.\n"
        f"Expected: {EXPECTED_PARAMETER_COUNT}\n"
        f"Actual: {total_parameters}"
    )


if trainable_parameters != total_parameters:

    raise RuntimeError(
        "Fusion model contains unexpected frozen parameters."
    )


print(
    "Parameter count: PASS"
)


# =============================================================================
# 17. FORWARD-PASS AUDIT
# =============================================================================

print("\nFORWARD-PASS AUDIT")
print("-" * 80)

BATCH_SIZE = 32

dummy_cxr = torch.randn(
    BATCH_SIZE,
    CXR_INPUT_DIM,
    dtype=torch.float32
)

dummy_genomic = torch.randn(
    BATCH_SIZE,
    GENOMIC_INPUT_DIM,
    dtype=torch.float32
)

model.eval()

with torch.no_grad():

    output = model(
        dummy_cxr,
        dummy_genomic
    )


logits = output[
    "logits"
]

cxr_representation = output[
    "cxr_representation"
]

genomic_representation = output[
    "genomic_representation"
]

fused_representation = output[
    "fused_representation"
]


print(
    "CXR input:",
    tuple(
        dummy_cxr.shape
    )
)

print(
    "Genomic input:",
    tuple(
        dummy_genomic.shape
    )
)

print(
    "CXR projected:",
    tuple(
        cxr_representation.shape
    )
)

print(
    "Genomic projected:",
    tuple(
        genomic_representation.shape
    )
)

print(
    "Fused:",
    tuple(
        fused_representation.shape
    )
)

print(
    "Logits:",
    tuple(
        logits.shape
    )
)


if cxr_representation.shape != (
    BATCH_SIZE,
    256
):

    raise RuntimeError(
        "CXR projection shape incorrect."
    )


if genomic_representation.shape != (
    BATCH_SIZE,
    128
):

    raise RuntimeError(
        "Genomic projection shape incorrect."
    )


if fused_representation.shape != (
    BATCH_SIZE,
    384
):

    raise RuntimeError(
        "Fusion representation shape incorrect."
    )


if logits.shape != (
    BATCH_SIZE,
    2
):

    raise RuntimeError(
        "Logit shape incorrect."
    )


if not torch.isfinite(
    logits
).all():

    raise RuntimeError(
        "Non-finite logits detected."
    )


print(
    "Forward pass: PASS"
)


# =============================================================================
# 18. REAL EMBEDDING FORWARD-PASS
# =============================================================================

print("\nREAL EMBEDDING FORWARD-PASS")
print("-" * 80)

sample_count = min(
    32,
    len(
        cxr_embeddings
    )
)

real_cxr_sample = torch.from_numpy(
    cxr_embeddings[
        :sample_count
    ]
).float()

real_genomic_sample = torch.from_numpy(
    genomic_embeddings[
        :sample_count
    ]
).float()


model.eval()

with torch.no_grad():

    real_output = model(
        real_cxr_sample,
        real_genomic_sample
    )


real_logits = real_output[
    "logits"
]


if not torch.isfinite(
    real_logits
).all():

    raise RuntimeError(
        "Non-finite logits from real embeddings."
    )


print(
    "Real CXR sample:",
    tuple(
        real_cxr_sample.shape
    )
)

print(
    "Real genomic sample:",
    tuple(
        real_genomic_sample.shape
    )
)

print(
    "Real multimodal logits:",
    tuple(
        real_logits.shape
    )
)

print(
    "Real embedding forward pass: PASS"
)


# =============================================================================
# 19. TRAINING CONFIGURATION — AUDIT ONLY
# =============================================================================
#
# These values are recorded here but NO optimizer is created
# and NO training is performed in Cell 3.
#
# The actual training will occur in Cell 4.
# =============================================================================

TRAINING_EPOCHS = 40
BATCH_SIZE_TRAIN = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.02
GRADIENT_CLIP = 1.0

EARLY_STOPPING = False

CHECKPOINT_SELECTION_METRIC = (
    "validation_roc_auc"
)

TEST_USED_FOR_TRAINING = False
TEST_USED_FOR_CHECKPOINT_SELECTION = False
TEST_USED_FOR_THRESHOLD_TUNING = False


print("\nTRAINING CONFIGURATION — AUDIT ONLY")
print("-" * 80)

print(
    "Fixed epochs:",
    TRAINING_EPOCHS
)

print(
    "Batch size:",
    BATCH_SIZE_TRAIN
)

print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Weight decay:",
    WEIGHT_DECAY
)

print(
    "Label smoothing:",
    LABEL_SMOOTHING
)

print(
    "Gradient clipping:",
    GRADIENT_CLIP
)

print(
    "Early stopping:",
    "DISABLED"
)

print(
    "Checkpoint selection:",
    CHECKPOINT_SELECTION_METRIC
)

print(
    "Test used for training:",
    "NO"
)

print(
    "Test used for checkpoint selection:",
    "NO"
)

print(
    "Test used for threshold tuning:",
    "NO"
)


# =============================================================================
# 20. MODEL ARCHITECTURE SUMMARY
# =============================================================================

architecture_summary = {

    "status":
        "PASS",

    "cxr_input_dimension":
        768,

    "cxr_projection_dimension":
        256,

    "genomic_input_dimension":
        128,

    "genomic_projection_dimension":
        128,

    "fusion_input_dimension":
        384,

    "fusion_hidden_dimension":
        128,

    "num_classes":
        2,

    "dropout":
        0.20,

    "total_parameters":
        total_parameters,

    "trainable_parameters":
        trainable_parameters,

    "fixed_epochs":
        TRAINING_EPOCHS,

    "batch_size":
        BATCH_SIZE_TRAIN,

    "learning_rate":
        LEARNING_RATE,

    "weight_decay":
        WEIGHT_DECAY,

    "label_smoothing":
        LABEL_SMOOTHING,

    "gradient_clipping":
        GRADIENT_CLIP,

    "early_stopping":
        False,

    "checkpoint_selection":
        CHECKPOINT_SELECTION_METRIC,

    "training_performed":
        False,

    "test_evaluation_performed":
        False,

    "test_used_for_training":
        False,

    "test_used_for_checkpoint_selection":
        False,

    "test_used_for_threshold_tuning":
        False,

    "frozen_cxr_encoder_epoch":
        7,

    "frozen_genomic_encoder_epoch":
        11,

}


ARCHITECTURE_FILE = (
    CELL3_ROOT
    / "Notebook15_Cell3_FINAL_Multimodal_Fusion_Architecture.json"
)

with open(
    ARCHITECTURE_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        architecture_summary,
        f,
        indent=4
    )


# =============================================================================
# 21. FINAL STATUS
# =============================================================================

print("\n")
print("=" * 80)
print("CELL 3 STATUS: PASS")
print("=" * 80)

print(
    "CXR input:",
    "768-D"
)

print(
    "CXR projection:",
    "256-D"
)

print(
    "Genomic input:",
    "128-D"
)

print(
    "Genomic projection:",
    "128-D"
)

print(
    "Fusion representation:",
    "384-D"
)

print(
    "Fusion hidden layer:",
    "128-D"
)

print(
    "Output classes:",
    2
)

print(
    "Total parameters:",
    total_parameters
)

print(
    "Fixed training epochs:",
    TRAINING_EPOCHS
)

print(
    "Early stopping:",
    "DISABLED"
)

print(
    "Checkpoint selection:",
    CHECKPOINT_SELECTION_METRIC
)

print(
    "Training performed:",
    "NO"
)

print(
    "Test evaluation:",
    "NO"
)

print(
    "\nArchitecture configuration:"
)

print(
    ARCHITECTURE_FILE
)

print(
    "\nSTOP HERE."
)

print(
    "Proceed to Cell 4 only if STATUS = PASS."
)

NOTEBOOK 15 — CELL 3
FINAL MULTIMODAL FUSION ARCHITECTURE AUDIT

PATH GATE
--------------------------------------------------------------------------------
Multimodal master             : PASS
Cell 1 summary                : PASS
CXR embeddings                : PASS
Genomic embeddings            : PASS
Targets                       : PASS
Condition IDs                 : PASS
Splits                        : PASS
Cell 2 summary                : PASS

DATASET SHAPE GATE
--------------------------------------------------------------------------------
Multimodal master: (1976, 5)
CXR embeddings: (1976, 768)
Genomic embeddings: (1976, 128)
Targets: (1976,)
Condition IDs: (1976,)
Splits: (1976,)
Dataset shape gate: PASS

FINITE-VALUE AUDIT
--------------------------------------------------------------------------------
CXR embeddings: finite
Genomic embeddings: finite
Targets: finite
Finite-value audit: PASS

CONDITION-ID AUDIT
-----------------------------------------------------------------

In [4]:
# =============================================================================
# NOTEBOOK 15 — CELL 4
# FINAL MULTIMODAL CXR + GENOMIC FUSION TRAINING
#
# CXR encoder     : FROZEN, Epoch 7
# Genomic encoder : FROZEN, Epoch 11
# Fusion network  : TRAINED
#
# Fixed epochs    : 40
# Early stopping  : DISABLED
# Checkpoint      : BEST VALIDATION ROC-AUC
# Test evaluation : NOT PERFORMED
# Test tuning     : NOT PERFORMED
# =============================================================================

from pathlib import Path
import json
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)


print("=" * 80)
print("NOTEBOOK 15 — CELL 4")
print("FINAL MULTIMODAL CXR + GENOMIC FUSION TRAINING")
print("=" * 80)


# =============================================================================
# 1. REPRODUCIBILITY
# =============================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Deterministic CPU behavior.
torch.use_deterministic_algorithms(
    False
)


# =============================================================================
# 2. PROJECT PATHS
# =============================================================================

PROJECT_ROOT = Path(
    r"C:\TBP\Metadata"
)

FINAL_ROOT = (
    PROJECT_ROOT
    / "Step_3B_8_CXR_CoAtNet_Preparation"
    / "QC"
    / "Clean_PSPNet_CXR_Cohort"
    / "FINAL_FROZEN_CLEAN_COHORT"
)

MULTIMODAL_ROOT = (
    FINAL_ROOT
    / "Multimodal_15"
)

CELL1_ROOT = (
    MULTIMODAL_ROOT
    / "Cell1_Alignment_Audit"
)

CELL2_ROOT = (
    MULTIMODAL_ROOT
    / "Cell2_Frozen_Embedding_Extraction"
)

CELL3_ROOT = (
    MULTIMODAL_ROOT
    / "Cell3_Fusion_Architecture"
)

CELL4_ROOT = (
    MULTIMODAL_ROOT
    / "Cell4_Fusion_Training"
)

CHECKPOINT_ROOT = (
    CELL4_ROOT
    / "checkpoints"
)

CELL4_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

CHECKPOINT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 3. INPUT FILES
# =============================================================================

MULTIMODAL_MASTER = (
    CELL1_ROOT
    / "Notebook15_Cell1_FINAL_1976_Multimodal_Master.csv"
)

CELL1_SUMMARY = (
    CELL1_ROOT
    / "Notebook15_Cell1_Multimodal_Alignment_Audit_Summary.json"
)

CXR_EMBEDDINGS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_CXR_Embeddings_768D.npy"
)

GENOMIC_EMBEDDINGS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_Genomic_Embeddings_128D.npy"
)

TARGETS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_Targets.npy"
)

CONDITION_IDS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_Condition_IDs.npy"
)

SPLITS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_Splits.npy"
)

CELL2_SUMMARY = (
    CELL2_ROOT
    / "Notebook15_Cell2_Frozen_Embedding_Extraction_Summary.json"
)

CELL3_ARCHITECTURE = (
    CELL3_ROOT
    / "Notebook15_Cell3_FINAL_Multimodal_Fusion_Architecture.json"
)


# =============================================================================
# 4. OUTPUT FILES
# =============================================================================

BEST_CHECKPOINT = (
    CHECKPOINT_ROOT
    / "Notebook15_Cell4_Best_Multimodal_Validation_ROC_AUC.pth"
)

TRAINING_HISTORY_FILE = (
    CELL4_ROOT
    / "Notebook15_Cell4_Multimodal_Training_History.csv"
)

TRAINING_SUMMARY_FILE = (
    CELL4_ROOT
    / "Notebook15_Cell4_Multimodal_Training_Summary.json"
)


# =============================================================================
# 5. PATH GATE
# =============================================================================

print("\nPATH GATE")
print("-" * 80)

required_paths = {

    "Multimodal master":
        MULTIMODAL_MASTER,

    "Cell 1 summary":
        CELL1_SUMMARY,

    "CXR embeddings":
        CXR_EMBEDDINGS_FILE,

    "Genomic embeddings":
        GENOMIC_EMBEDDINGS_FILE,

    "Targets":
        TARGETS_FILE,

    "Condition IDs":
        CONDITION_IDS_FILE,

    "Splits":
        SPLITS_FILE,

    "Cell 2 summary":
        CELL2_SUMMARY,

    "Cell 3 architecture":
        CELL3_ARCHITECTURE,

}

for name, path in required_paths.items():

    if not path.exists():

        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

    print(
        f"{name:<30}: PASS"
    )


# =============================================================================
# 6. LOAD AND VERIFY PREVIOUS CELL SUMMARIES
# =============================================================================

with open(
    CELL1_SUMMARY,
    "r",
    encoding="utf-8"
) as f:

    cell1_summary = json.load(f)


with open(
    CELL2_SUMMARY,
    "r",
    encoding="utf-8"
) as f:

    cell2_summary = json.load(f)


with open(
    CELL3_ARCHITECTURE,
    "r",
    encoding="utf-8"
) as f:

    cell3_summary = json.load(f)


if cell1_summary.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Cell 1 did not PASS."
    )


if cell2_summary.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Cell 2 did not PASS."
    )


if cell3_summary.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Cell 3 did not PASS."
    )


if cell2_summary.get(
    "training_performed"
) is not False:

    raise RuntimeError(
        "Cell 2 reports training was performed."
    )


if cell2_summary.get(
    "fine_tuning_performed"
) is not False:

    raise RuntimeError(
        "Cell 2 reports fine-tuning was performed."
    )


print(
    "\nPrevious-cell validation: PASS"
)


# =============================================================================
# 7. LOAD DATA
# =============================================================================

multimodal_master = pd.read_csv(
    MULTIMODAL_MASTER,
    low_memory=False
)

cxr_embeddings = np.load(
    CXR_EMBEDDINGS_FILE
)

genomic_embeddings = np.load(
    GENOMIC_EMBEDDINGS_FILE
)

targets = np.load(
    TARGETS_FILE
)

condition_ids = np.load(
    CONDITION_IDS_FILE,
    allow_pickle=True
).astype(str)

splits = np.load(
    SPLITS_FILE,
    allow_pickle=True
).astype(str
)


# =============================================================================
# 8. DATASET INTEGRITY
# =============================================================================

print("\nFROZEN MULTIMODAL DATASET")
print("-" * 80)

print(
    "Full:",
    len(condition_ids)
)

print(
    "CXR embeddings:",
    cxr_embeddings.shape
)

print(
    "Genomic embeddings:",
    genomic_embeddings.shape
)


if len(condition_ids) != 1976:

    raise RuntimeError(
        "Expected 1976 conditions."
    )


if cxr_embeddings.shape != (
    1976,
    768
):

    raise RuntimeError(
        "Unexpected CXR embedding shape."
    )


if genomic_embeddings.shape != (
    1976,
    128
):

    raise RuntimeError(
        "Unexpected genomic embedding shape."
    )


if targets.shape != (
    1976,
):

    raise RuntimeError(
        "Unexpected target shape."
    )


if splits.shape != (
    1976,
):

    raise RuntimeError(
        "Unexpected split shape."
    )


if len(
    set(condition_ids)
) != 1976:

    raise RuntimeError(
        "Condition IDs are not unique."
    )


if not np.isfinite(
    cxr_embeddings
).all():

    raise RuntimeError(
        "Non-finite CXR embeddings."
    )


if not np.isfinite(
    genomic_embeddings
).all():

    raise RuntimeError(
        "Non-finite genomic embeddings."
    )


# =============================================================================
# 9. TARGET AND SPLIT GATES
# =============================================================================

expected_counts = {

    "train":
        1185,

    "validation":
        395,

    "test":
        396,

}

expected_targets = {

    "train":
        (422, 763),

    "validation":
        (141, 254),

    "test":
        (141, 255),

}


print("\nSPLIT GATE")
print("-" * 80)

for split_name in [
    "train",
    "validation",
    "test"
]:

    mask = (
        splits
        ==
        split_name
    )

    count = int(
        mask.sum()
    )

    ds_count = int(
        (
            targets[mask]
            ==
            0
        ).sum()
    )

    dr_count = int(
        (
            targets[mask]
            ==
            1
        ).sum()
    )

    print(
        f"{split_name:<12}: "
        f"{count} | "
        f"DS={ds_count} | "
        f"DR={dr_count}"
    )

    if count != expected_counts[
        split_name
    ]:

        raise RuntimeError(
            f"{split_name} count mismatch."
        )

    if (
        ds_count,
        dr_count
    ) != expected_targets[
        split_name
    ]:

        raise RuntimeError(
            f"{split_name} target distribution mismatch."
        )


# =============================================================================
# 10. SPLIT OVERLAP
# =============================================================================

train_ids = set(
    condition_ids[
        splits == "train"
    ]
)

validation_ids = set(
    condition_ids[
        splits == "validation"
    ]
)

test_ids = set(
    condition_ids[
        splits == "test"
    ]
)


if train_ids & validation_ids:

    raise RuntimeError(
        "Train/validation overlap detected."
    )


if train_ids & test_ids:

    raise RuntimeError(
        "Train/test overlap detected."
    )


if validation_ids & test_ids:

    raise RuntimeError(
        "Validation/test overlap detected."
    )


print(
    "Split overlap: PASS"
)


# =============================================================================
# 11. TEST ISOLATION GATE
# =============================================================================

print("\nTEST ISOLATION")
print("-" * 80)

print(
    "Test embeddings loaded for:",
    "structural isolation only"
)

print(
    "Test used for training:",
    "NO"
)

print(
    "Test used for checkpoint selection:",
    "NO"
)

print(
    "Test used for threshold tuning:",
    "NO"
)

print(
    "Test predictions generated:",
    "NO"
)


# The test embeddings are intentionally not included in
# the training/validation DataLoaders below.


# =============================================================================
# 12. MULTIMODAL DATASET
# =============================================================================

class MultimodalEmbeddingDataset(
    Dataset
):

    def __init__(
        self,
        cxr,
        genomic,
        targets,
        condition_ids,
        indices
    ):

        self.cxr = cxr[
            indices
        ].astype(
            np.float32,
            copy=False
        )

        self.genomic = genomic[
            indices
        ].astype(
            np.float32,
            copy=False
        )

        self.targets = targets[
            indices
        ].astype(
            np.int64,
            copy=False
        )

        self.condition_ids = (
            condition_ids[
                indices
            ]
        )

    def __len__(
        self
    ):

        return len(
            self.targets
        )

    def __getitem__(
        self,
        index
    ):

        return {

            "cxr":
                torch.from_numpy(
                    self.cxr[index]
                ),

            "genomic":
                torch.from_numpy(
                    self.genomic[index]
                ),

            "target":
                torch.tensor(
                    self.targets[index],
                    dtype=torch.long
                ),

            "condition_id":
                self.condition_ids[index],

        }


# =============================================================================
# 13. TRAIN / VALIDATION INDICES
# =============================================================================

train_indices = np.where(
    splits == "train"
)[0]

validation_indices = np.where(
    splits == "validation"
)[0]

test_indices = np.where(
    splits == "test"
)[0]


if len(
    train_indices
) != 1185:

    raise RuntimeError(
        "Training index count incorrect."
    )


if len(
    validation_indices
) != 395:

    raise RuntimeError(
        "Validation index count incorrect."
    )


if len(
    test_indices
) != 396:

    raise RuntimeError(
        "Test index count incorrect."
    )


# =============================================================================
# 14. DATASETS
# =============================================================================

train_dataset = MultimodalEmbeddingDataset(
    cxr_embeddings,
    genomic_embeddings,
    targets,
    condition_ids,
    train_indices
)

validation_dataset = MultimodalEmbeddingDataset(
    cxr_embeddings,
    genomic_embeddings,
    targets,
    condition_ids,
    validation_indices
)


# =============================================================================
# 15. DATALOADERS
# =============================================================================

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)


print("\nDATALOADERS")
print("-" * 80)

print(
    "Training samples:",
    len(train_dataset)
)

print(
    "Validation samples:",
    len(validation_dataset)
)

print(
    "Training batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(validation_loader)
)

print(
    "Test dataset created:",
    "NO"
)


# =============================================================================
# 16. MODEL
# =============================================================================

class MultimodalFusionModel(
    nn.Module
):

    def __init__(
        self,
        cxr_input_dim=768,
        cxr_projection_dim=256,
        genomic_input_dim=128,
        genomic_projection_dim=128,
        fusion_hidden_dim=128,
        num_classes=2,
        dropout=0.20
    ):

        super().__init__()

        self.cxr_projection = nn.Sequential(

            nn.Linear(
                cxr_input_dim,
                cxr_projection_dim
            ),

            nn.LayerNorm(
                cxr_projection_dim
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            )
        )

        self.genomic_projection = nn.Sequential(

            nn.Linear(
                genomic_input_dim,
                genomic_projection_dim
            ),

            nn.LayerNorm(
                genomic_projection_dim
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            )
        )

        self.fusion = nn.Sequential(

            nn.Linear(
                cxr_projection_dim
                +
                genomic_projection_dim,
                fusion_hidden_dim
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                fusion_hidden_dim,
                num_classes
            )
        )

    def forward(
        self,
        cxr_embedding,
        genomic_embedding
    ):

        cxr_representation = (
            self.cxr_projection(
                cxr_embedding
            )
        )

        genomic_representation = (
            self.genomic_projection(
                genomic_embedding
            )
        )

        fused_representation = torch.cat(
            [
                cxr_representation,
                genomic_representation
            ],
            dim=1
        )

        logits = self.fusion(
            fused_representation
        )

        return logits


# =============================================================================
# 17. MODEL CREATION
# =============================================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nDEVICE")
print("-" * 80)

print(
    "Device:",
    device
)


model = MultimodalFusionModel(
    cxr_input_dim=768,
    cxr_projection_dim=256,
    genomic_input_dim=128,
    genomic_projection_dim=128,
    fusion_hidden_dim=128,
    num_classes=2,
    dropout=0.20
).to(device)


# =============================================================================
# 18. PARAMETER GATE
# =============================================================================

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

EXPECTED_PARAMETERS = 263682

print(
    "Total parameters:",
    total_parameters
)

print(
    "Trainable parameters:",
    trainable_parameters
)

if total_parameters != EXPECTED_PARAMETERS:

    raise RuntimeError(
        f"Expected {EXPECTED_PARAMETERS} parameters, "
        f"found {total_parameters}."
    )

if trainable_parameters != EXPECTED_PARAMETERS:

    raise RuntimeError(
        "Unexpected frozen parameters in fusion network."
    )


print(
    "Parameter gate: PASS"
)


# =============================================================================
# 19. CLASS WEIGHTS
# =============================================================================
#
# Class weights are calculated from TRAINING DATA ONLY.
# Validation and test are never used to calculate weights.
# =============================================================================

train_targets = targets[
    train_indices
]

train_ds_count = int(
    (
        train_targets
        ==
        0
    ).sum()
)

train_dr_count = int(
    (
        train_targets
        ==
        1
    ).sum()
)

total_train = (
    train_ds_count
    +
    train_dr_count
)

ds_weight = (
    total_train
    /
    (
        2.0
        *
        train_ds_count
    )
)

dr_weight = (
    total_train
    /
    (
        2.0
        *
        train_dr_count
    )
)

class_weights = torch.tensor(
    [
        ds_weight,
        dr_weight
    ],
    dtype=torch.float32,
    device=device
)


print("\nCLASS WEIGHTS")
print("-" * 80)

print(
    "DS weight:",
    float(
        class_weights[0]
    )
)

print(
    "DR weight:",
    float(
        class_weights[1]
    )

)

# =============================================================================
# 20. LOSS FUNCTION
# =============================================================================

LABEL_SMOOTHING = 0.02

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=LABEL_SMOOTHING
)


# =============================================================================
# 21. OPTIMIZER
# =============================================================================

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# =============================================================================
# 22. SCHEDULER
# =============================================================================

EPOCHS = 40

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-6
)


# =============================================================================
# 23. TRAINING HELPER
# =============================================================================

def calculate_binary_metrics(
    targets_array,
    probabilities
):

    predictions = (
        probabilities
        >=
        0.5
    ).astype(
        np.int64
    )

    accuracy = accuracy_score(
        targets_array,
        predictions
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            targets_array,
            predictions
        )
    )

    precision = precision_score(
        targets_array,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        targets_array,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        targets_array,
        predictions,
        zero_division=0
    )

    try:

        roc_auc = roc_auc_score(
            targets_array,
            probabilities
        )

    except ValueError:

        roc_auc = float(
            "nan"
        )

    try:

        pr_auc = average_precision_score(
            targets_array,
            probabilities
        )

    except ValueError:

        pr_auc = float(
            "nan"
        )

    return {

        "accuracy":
            float(accuracy),

        "balanced_accuracy":
            float(balanced_accuracy),

        "precision":
            float(precision),

        "recall":
            float(recall),

        "f1":
            float(f1),

        "roc_auc":
            float(roc_auc),

        "pr_auc":
            float(pr_auc),

    }


# =============================================================================
# 24. TRAINING FUNCTION
# =============================================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    running_loss = 0.0
    all_targets = []
    all_probabilities = []

    for batch in loader:

        cxr_batch = batch[
            "cxr"
        ].to(device)

        genomic_batch = batch[
            "genomic"
        ].to(device)

        target_batch = batch[
            "target"
        ].to(device)

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(
            cxr_batch,
            genomic_batch
        )

        loss = criterion(
            logits,
            target_batch
        )

        if not torch.isfinite(
            loss
        ):

            raise RuntimeError(
                "Non-finite training loss detected."
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        running_loss += (
            loss.item()
            *
            target_batch.size(0)
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )[:, 1]

        all_targets.extend(
            target_batch.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        all_probabilities.extend(
            probabilities.detach()
            .cpu()
            .numpy()
            .tolist()
        )

    epoch_loss = (
        running_loss
        /
        len(loader.dataset)
    )

    metrics = calculate_binary_metrics(
        np.asarray(
            all_targets
        ),
        np.asarray(
            all_probabilities
        )
    )

    return epoch_loss, metrics


# =============================================================================
# 25. VALIDATION FUNCTION
# =============================================================================

def evaluate_validation(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0
    all_targets = []
    all_probabilities = []

    with torch.no_grad():

        for batch in loader:

            cxr_batch = batch[
                "cxr"
            ].to(device)

            genomic_batch = batch[
                "genomic"
            ].to(device)

            target_batch = batch[
                "target"
            ].to(device)

            logits = model(
                cxr_batch,
                genomic_batch
            )

            loss = criterion(
                logits,
                target_batch
            )

            if not torch.isfinite(
                loss
            ):

                raise RuntimeError(
                    "Non-finite validation loss detected."
                )

            running_loss += (
                loss.item()
                *
                target_batch.size(0)
            )

            probabilities = torch.softmax(
                logits,
                dim=1
            )[:, 1]

            all_targets.extend(
                target_batch.cpu()
                .numpy()
                .tolist()
            )

            all_probabilities.extend(
                probabilities.cpu()
                .numpy()
                .tolist()
            )

    epoch_loss = (
        running_loss
        /
        len(loader.dataset)
    )

    metrics = calculate_binary_metrics(
        np.asarray(
            all_targets
        ),
        np.asarray(
            all_probabilities
        )
    )

    return epoch_loss, metrics


# =============================================================================
# 26. FIXED-EPOCH TRAINING
# =============================================================================

print("\n")
print("=" * 80)
print("STARTING FINAL MULTIMODAL FUSION TRAINING")
print("=" * 80)

print(
    "Epochs:",
    EPOCHS
)

print(
    "Early stopping:",
    "DISABLED"
)

print(
    "Checkpoint selection:",
    "VALIDATION ROC-AUC"
)

print(
    "Test evaluation:",
    "NOT PERFORMED"
)

print(
    "Test threshold tuning:",
    "NOT PERFORMED"
)


best_val_auc = -np.inf
best_epoch = None
best_val_accuracy = None
best_val_f1 = None

history = []


for epoch in range(
    1,
    EPOCHS + 1
):

    train_loss, train_metrics = (
        train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device
        )
    )

    validation_loss, validation_metrics = (
        evaluate_validation(
            model,
            validation_loader,
            criterion,
            device
        )
    )

    current_lr = (
        optimizer.param_groups[0][
            "lr"
        ]
    )

    current_val_auc = (
        validation_metrics[
            "roc_auc"
        ]
    )

    is_best = (
        current_val_auc
        >
        best_val_auc
    )

    if is_best:

        best_val_auc = (
            current_val_auc
        )

        best_epoch = epoch

        best_val_accuracy = (
            validation_metrics[
                "accuracy"
            ]
        )

        best_val_f1 = (
            validation_metrics[
                "f1"
            ]
        )

        checkpoint = {

            "epoch":
                epoch,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "scheduler_state_dict":
                scheduler.state_dict(),

            "best_validation_roc_auc":
                best_val_auc,

            "validation_accuracy":
                best_val_accuracy,

            "validation_f1":
                best_val_f1,

            "seed":
                SEED,

            "cxr_encoder_epoch":
                7,

            "genomic_encoder_epoch":
                11,

            "architecture":
                "MultimodalFusionModel",

            "cxr_input_dim":
                768,

            "cxr_projection_dim":
                256,

            "genomic_input_dim":
                128,

            "genomic_projection_dim":
                128,

            "fusion_hidden_dim":
                128,

            "dropout":
                0.20,

            "epochs":
                EPOCHS,

            "early_stopping":
                False,

            "test_used":
                False,

        }

        torch.save(
            checkpoint,
            BEST_CHECKPOINT
        )

    history_row = {

        "epoch":
            epoch,

        "train_loss":
            train_loss,

        "train_accuracy":
            train_metrics[
                "accuracy"
            ],

        "train_balanced_accuracy":
            train_metrics[
                "balanced_accuracy"
            ],

        "train_precision":
            train_metrics[
                "precision"
            ],

        "train_recall":
            train_metrics[
                "recall"
            ],

        "train_f1":
            train_metrics[
                "f1"
            ],

        "train_roc_auc":
            train_metrics[
                "roc_auc"
            ],

        "train_pr_auc":
            train_metrics[
                "pr_auc"
            ],

        "validation_loss":
            validation_loss,

        "validation_accuracy":
            validation_metrics[
                "accuracy"
            ],

        "validation_balanced_accuracy":
            validation_metrics[
                "balanced_accuracy"
            ],

        "validation_precision":
            validation_metrics[
                "precision"
            ],

        "validation_recall":
            validation_metrics[
                "recall"
            ],

        "validation_f1":
            validation_metrics[
                "f1"
            ],

        "validation_roc_auc":
            validation_metrics[
                "roc_auc"
            ],

        "validation_pr_auc":
            validation_metrics[
                "pr_auc"
            ],

        "learning_rate":
            current_lr,

        "best_so_far":
            bool(is_best),

    }

    history.append(
        history_row
    )

    marker = (
        " *BEST*"
        if is_best
        else ""
    )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss {train_loss:.4f} | "
        f"Train Acc {train_metrics['accuracy']:.4f} | "
        f"Val Loss {validation_loss:.4f} | "
        f"Val Acc {validation_metrics['accuracy']:.4f} | "
        f"Val BalAcc {validation_metrics['balanced_accuracy']:.4f} | "
        f"Val F1 {validation_metrics['f1']:.4f} | "
        f"Val AUC {validation_metrics['roc_auc']:.4f} | "
        f"Val PR-AUC {validation_metrics['pr_auc']:.4f} | "
        f"LR {current_lr:.2e}"
        f"{marker}"
    )

    scheduler.step()


# =============================================================================
# 27. TRAINING HISTORY
# =============================================================================

history_df = pd.DataFrame(
    history
)

history_df.to_csv(
    TRAINING_HISTORY_FILE,
    index=False
)


# =============================================================================
# 28. VERIFY CHECKPOINT
# =============================================================================

if not BEST_CHECKPOINT.exists():

    raise RuntimeError(
        "Best validation checkpoint was not created."
    )


if best_epoch is None:

    raise RuntimeError(
        "No best validation epoch was selected."
    )


# =============================================================================
# 29. FINAL TRAINING SUMMARY
# =============================================================================

training_summary = {

    "status":
        "PASS",

    "epochs_completed":
        EPOCHS,

    "early_stopping":
        False,

    "best_epoch":
        int(best_epoch),

    "best_validation_roc_auc":
        float(best_val_auc),

    "best_validation_accuracy":
        float(best_val_accuracy),

    "best_validation_f1":
        float(best_val_f1),

    "train_conditions":
        1185,

    "validation_conditions":
        395,

    "test_conditions":
        396,

    "cxr_encoder_checkpoint_epoch":
        7,

    "genomic_encoder_checkpoint_epoch":
        11,

    "fusion_parameters":
        total_parameters,

    "batch_size":
        BATCH_SIZE,

    "learning_rate":
        LEARNING_RATE,

    "weight_decay":
        WEIGHT_DECAY,

    "label_smoothing":
        LABEL_SMOOTHING,

    "gradient_clipping":
        1.0,

    "scheduler":
        "CosineAnnealingLR",

    "checkpoint_selection_metric":
        "validation_roc_auc",

    "test_used_for_training":
        False,

    "test_used_for_checkpoint_selection":
        False,

    "test_used_for_threshold_tuning":
        False,

    "test_predictions_generated":
        False,

    "best_checkpoint":
        str(
            BEST_CHECKPOINT
        ),

    "training_history":
        str(
            TRAINING_HISTORY_FILE
        )

}


with open(
    TRAINING_SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        training_summary,
        f,
        indent=4
    )


# =============================================================================
# 30. FINAL STATUS
# =============================================================================

print("\n")
print("=" * 80)
print("CELL 4 STATUS: PASS")
print("=" * 80)

print(
    "Epochs completed:",
    EPOCHS
)

print(
    "Early stopping:",
    "DISABLED"
)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best validation ROC-AUC:",
    f"{best_val_auc:.6f}"
)

print(
    "Validation accuracy at best AUC:",
    f"{best_val_accuracy:.6f}"
)

print(
    "Validation F1 at best AUC:",
    f"{best_val_f1:.6f}"
)

print(
    "Test used:",
    "NO"
)

print(
    "Test predictions:",
    "NO"
)

print(
    "\nBest checkpoint:"
)

print(
    BEST_CHECKPOINT
)

print(
    "\nTraining history:"
)

print(
    TRAINING_HISTORY_FILE
)

print(
    "\nTraining summary:"
)

print(
    TRAINING_SUMMARY_FILE
)

print(
    "\nSTOP HERE."
)

print(
    "Do not evaluate the frozen test set until the training output "
    "has been reviewed."
)

NOTEBOOK 15 — CELL 4
FINAL MULTIMODAL CXR + GENOMIC FUSION TRAINING

PATH GATE
--------------------------------------------------------------------------------
Multimodal master             : PASS
Cell 1 summary                : PASS
CXR embeddings                : PASS
Genomic embeddings            : PASS
Targets                       : PASS
Condition IDs                 : PASS
Splits                        : PASS
Cell 2 summary                : PASS
Cell 3 architecture           : PASS

Previous-cell validation: PASS

FROZEN MULTIMODAL DATASET
--------------------------------------------------------------------------------
Full: 1976
CXR embeddings: (1976, 768)
Genomic embeddings: (1976, 128)

SPLIT GATE
--------------------------------------------------------------------------------
train       : 1185 | DS=422 | DR=763
validation  : 395 | DS=141 | DR=254
test        : 396 | DS=141 | DR=255
Split overlap: PASS

TEST ISOLATION
----------------------------------------------------------

In [5]:
# =============================================================================
# NOTEBOOK 15 — CELL 5
# FINAL MULTIMODAL BEST-CHECKPOINT VALIDATION REVIEW
#
# Purpose:
#   Independently reload the saved best fusion checkpoint and reproduce
#   validation metrics before frozen test evaluation.
#
# Training                  : NO
# Test evaluation           : NO
# Threshold tuning          : NO
# Checkpoint selection      : ALREADY COMPLETED IN CELL 4
# Test data loaded          : NO
# =============================================================================

from pathlib import Path
import json
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)


print("=" * 80)
print("NOTEBOOK 15 — CELL 5")
print("FINAL MULTIMODAL BEST-CHECKPOINT VALIDATION REVIEW")
print("=" * 80)


# =============================================================================
# 1. REPRODUCIBILITY
# =============================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# =============================================================================
# 2. PROJECT PATHS
# =============================================================================

PROJECT_ROOT = Path(
    r"C:\TBP\Metadata"
)

FINAL_ROOT = (
    PROJECT_ROOT
    / "Step_3B_8_CXR_CoAtNet_Preparation"
    / "QC"
    / "Clean_PSPNet_CXR_CohORT"
    / "FINAL_FROZEN_CLEAN_COHORT"
)

# Correct the folder name if the previous path does not exist.
if not FINAL_ROOT.exists():

    FINAL_ROOT = (
        PROJECT_ROOT
        / "Step_3B_8_CXR_CoAtNet_Preparation"
        / "QC"
        / "Clean_PSPNet_CXR_Cohort"
        / "FINAL_FROZEN_CLEAN_COHORT"
    )


MULTIMODAL_ROOT = (
    FINAL_ROOT
    / "Multimodal_15"
)

CELL1_ROOT = (
    MULTIMODAL_ROOT
    / "Cell1_Alignment_Audit"
)

CELL2_ROOT = (
    MULTIMODAL_ROOT
    / "Cell2_Frozen_Embedding_Extraction"
)

CELL4_ROOT = (
    MULTIMODAL_ROOT
    / "Cell4_Fusion_Training"
)

CELL5_ROOT = (
    MULTIMODAL_ROOT
    / "Cell5_Best_Checkpoint_Validation_Review"
)

CELL5_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 3. INPUT FILES
# =============================================================================

MULTIMODAL_MASTER = (
    CELL1_ROOT
    / "Notebook15_Cell1_FINAL_1976_Multimodal_Master.csv"
)

CELL1_SUMMARY = (
    CELL1_ROOT
    / "Notebook15_Cell1_Multimodal_Alignment_Audit_Summary.json"
)

CXR_EMBEDDINGS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_CXR_Embeddings_768D.npy"
)

GENOMIC_EMBEDDINGS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_Genomic_Embeddings_128D.npy"
)

TARGETS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_Targets.npy"
)

CONDITION_IDS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_Condition_IDs.npy"
)

SPLITS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_Splits.npy"
)

CELL2_SUMMARY = (
    CELL2_ROOT
    / "Notebook15_Cell2_Frozen_Embedding_Extraction_Summary.json"
)

BEST_CHECKPOINT = (
    CELL4_ROOT
    / "checkpoints"
    / "Notebook15_Cell4_Best_Multimodal_Validation_ROC_AUC.pth"
)

CELL4_SUMMARY = (
    CELL4_ROOT
    / "Notebook15_Cell4_Multimodal_Training_Summary.json"
)

CELL4_HISTORY = (
    CELL4_ROOT
    / "Notebook15_Cell4_Multimodal_Training_History.csv"
)


# =============================================================================
# 4. OUTPUT FILES
# =============================================================================

VALIDATION_PREDICTIONS_FILE = (
    CELL5_ROOT
    / "Notebook15_Cell5_Best_Checkpoint_Validation_Predictions.csv"
)

VALIDATION_METRICS_FILE = (
    CELL5_ROOT
    / "Notebook15_Cell5_Best_Checkpoint_Validation_Metrics.json"
)

VALIDATION_CONFUSION_MATRIX_FILE = (
    CELL5_ROOT
    / "Notebook15_Cell5_Best_Checkpoint_Validation_Confusion_Matrix.csv"
)

CELL5_SUMMARY_FILE = (
    CELL5_ROOT
    / "Notebook15_Cell5_Best_Checkpoint_Validation_Review_Summary.json"
)


# =============================================================================
# 5. PATH GATE
# =============================================================================

print("\nPATH GATE")
print("-" * 80)

required_paths = {

    "Multimodal master":
        MULTIMODAL_MASTER,

    "Cell 1 summary":
        CELL1_SUMMARY,

    "CXR embeddings":
        CXR_EMBEDDINGS_FILE,

    "Genomic embeddings":
        GENOMIC_EMBEDDINGS_FILE,

    "Targets":
        TARGETS_FILE,

    "Condition IDs":
        CONDITION_IDS_FILE,

    "Splits":
        SPLITS_FILE,

    "Cell 2 summary":
        CELL2_SUMMARY,

    "Best fusion checkpoint":
        BEST_CHECKPOINT,

    "Cell 4 summary":
        CELL4_SUMMARY,

    "Cell 4 history":
        CELL4_HISTORY,

}

for name, path in required_paths.items():

    if not path.exists():

        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

    print(
        f"{name:<32}: PASS"
    )


# =============================================================================
# 6. LOAD PREVIOUS SUMMARIES
# =============================================================================

with open(
    CELL1_SUMMARY,
    "r",
    encoding="utf-8"
) as f:

    cell1_summary = json.load(f)


with open(
    CELL2_SUMMARY,
    "r",
    encoding="utf-8"
) as f:

    cell2_summary = json.load(f)


with open(
    CELL4_SUMMARY,
    "r",
    encoding="utf-8"
) as f:

    cell4_summary = json.load(f)


cell4_history = pd.read_csv(
    CELL4_HISTORY
)


# =============================================================================
# 7. PREVIOUS CELL STATUS GATES
# =============================================================================

print("\nPREVIOUS CELL STATUS")
print("-" * 80)

if cell1_summary.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Cell 1 did not PASS."
    )

if cell2_summary.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Cell 2 did not PASS."
    )

if cell4_summary.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Cell 4 did not PASS."
    )

print(
    "Cell 1: PASS"
)

print(
    "Cell 2: PASS"
)

print(
    "Cell 4: PASS"
)


# =============================================================================
# 8. VERIFY CELL 4 TRAINING HISTORY
# =============================================================================

print("\nCELL 4 TRAINING HISTORY AUDIT")
print("-" * 80)

if len(
    cell4_history
) != 40:

    raise RuntimeError(
        f"Expected 40 training epochs, "
        f"found {len(cell4_history)}."
    )

best_history_row = cell4_history.loc[
    cell4_history[
        "validation_roc_auc"
    ].idxmax()
]

history_best_epoch = int(
    best_history_row[
        "epoch"
    ]
)

history_best_auc = float(
    best_history_row[
        "validation_roc_auc"
    ]
)

print(
    "Epochs recorded:",
    len(cell4_history)
)

print(
    "Best history epoch:",
    history_best_epoch
)

print(
    "Best history validation ROC-AUC:",
    f"{history_best_auc:.6f}"
)


# =============================================================================
# 9. VERIFY EXPECTED BEST CHECKPOINT
# =============================================================================

expected_best_epoch = int(
    cell4_summary[
        "best_epoch"
    ]
)

expected_best_auc = float(
    cell4_summary[
        "best_validation_roc_auc"
    ]
)

print(
    "Summary best epoch:",
    expected_best_epoch
)

print(
    "Summary best validation ROC-AUC:",
    f"{expected_best_auc:.6f}"
)

if history_best_epoch != expected_best_epoch:

    raise RuntimeError(
        "Training history and summary disagree on best epoch."
    )

if not np.isclose(
    history_best_auc,
    expected_best_auc,
    atol=1e-10
):

    raise RuntimeError(
        "Training history and summary disagree on best ROC-AUC."
    )

if expected_best_epoch != 1:

    raise RuntimeError(
        "Expected frozen multimodal best checkpoint to be Epoch 1 "
        f"based on Cell 4 output, but found Epoch {expected_best_epoch}."
    )

print(
    "Cell 4 checkpoint-selection audit: PASS"
)


# =============================================================================
# 10. LOAD FROZEN EMBEDDINGS
# =============================================================================

cxr_embeddings = np.load(
    CXR_EMBEDDINGS_FILE
)

genomic_embeddings = np.load(
    GENOMIC_EMBEDDINGS_FILE
)

targets = np.load(
    TARGETS_FILE
)

condition_ids = np.load(
    CONDITION_IDS_FILE,
    allow_pickle=True
).astype(str)

splits = np.load(
    SPLITS_FILE,
    allow_pickle=True
).astype(str)


# =============================================================================
# 11. DATASET SHAPE GATE
# =============================================================================

print("\nFROZEN EMBEDDING DATASET")
print("-" * 80)

print(
    "CXR:",
    cxr_embeddings.shape
)

print(
    "Genomic:",
    genomic_embeddings.shape
)

print(
    "Targets:",
    targets.shape
)

print(
    "Condition IDs:",
    condition_ids.shape
)

print(
    "Splits:",
    splits.shape
)


if cxr_embeddings.shape != (
    1976,
    768
):

    raise RuntimeError(
        "Unexpected CXR embedding shape."
    )

if genomic_embeddings.shape != (
    1976,
    128
):

    raise RuntimeError(
        "Unexpected genomic embedding shape."
    )

if targets.shape != (
    1976,
):

    raise RuntimeError(
        "Unexpected target shape."
    )

if condition_ids.shape != (
    1976,
):

    raise RuntimeError(
        "Unexpected condition-ID shape."
    )

if splits.shape != (
    1976,
):

    raise RuntimeError(
        "Unexpected split shape."
    )


# =============================================================================
# 12. VALIDATION-ONLY DATA SELECTION
# =============================================================================

validation_indices = np.where(
    splits == "validation"
)[0]

print(
    "Validation conditions:",
    len(validation_indices)
)

if len(
    validation_indices
) != 395:

    raise RuntimeError(
        "Validation set must contain exactly 395 conditions."
    )


validation_cxr = (
    cxr_embeddings[
        validation_indices
    ]
    .astype(
        np.float32,
        copy=False
    )
)

validation_genomic = (
    genomic_embeddings[
        validation_indices
    ]
    .astype(
        np.float32,
        copy=False
    )
)

validation_targets = (
    targets[
        validation_indices
    ]
    .astype(
        np.int64,
        copy=False
    )
)

validation_ids = (
    condition_ids[
        validation_indices
    ]
)


# =============================================================================
# 13. VALIDATION TARGET AUDIT
# =============================================================================

validation_ds = int(
    (
        validation_targets
        ==
        0
    ).sum()
)

validation_dr = int(
    (
        validation_targets
        ==
        1
    ).sum()
)

print(
    "Validation DS:",
    validation_ds
)

print(
    "Validation DR:",
    validation_dr
)

if validation_ds != 141:

    raise RuntimeError(
        "Validation DS count mismatch."
    )

if validation_dr != 254:

    raise RuntimeError(
        "Validation DR count mismatch."
    )

print(
    "Validation target distribution: PASS"
)


# =============================================================================
# 14. IMPORTANT TEST-ISOLATION GATE
# =============================================================================
#
# We intentionally DO NOT load:
#
#   test_indices
#   test embeddings
#   test targets
#
# for prediction.
#
# The frozen test set remains untouched.
# =============================================================================

print("\nTEST ISOLATION")
print("-" * 80)

print(
    "Test prediction generated:",
    "NO"
)

print(
    "Test threshold tuning:",
    "NO"
)

print(
    "Test checkpoint selection:",
    "NO"
)

print(
    "Test metrics calculated:",
    "NO"
)

print(
    "Test isolation: PASS"
)


# =============================================================================
# 15. MODEL DEFINITION
# =============================================================================

class MultimodalFusionModel(
    nn.Module
):

    def __init__(
        self,
        cxr_input_dim=768,
        cxr_projection_dim=256,
        genomic_input_dim=128,
        genomic_projection_dim=128,
        fusion_hidden_dim=128,
        num_classes=2,
        dropout=0.20
    ):

        super().__init__()

        self.cxr_projection = nn.Sequential(

            nn.Linear(
                cxr_input_dim,
                cxr_projection_dim
            ),

            nn.LayerNorm(
                cxr_projection_dim
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            )
        )

        self.genomic_projection = nn.Sequential(

            nn.Linear(
                genomic_input_dim,
                genomic_projection_dim
            ),

            nn.LayerNorm(
                genomic_projection_dim
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            )
        )

        self.fusion = nn.Sequential(

            nn.Linear(
                cxr_projection_dim
                +
                genomic_projection_dim,
                fusion_hidden_dim
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                fusion_hidden_dim,
                num_classes
            )
        )

    def forward(
        self,
        cxr_embedding,
        genomic_embedding
    ):

        cxr_representation = (
            self.cxr_projection(
                cxr_embedding
            )
        )

        genomic_representation = (
            self.genomic_projection(
                genomic_embedding
            )
        )

        fused_representation = torch.cat(
            [
                cxr_representation,
                genomic_representation
            ],
            dim=1
        )

        logits = self.fusion(
            fused_representation
        )

        return logits


# =============================================================================
# 16. MODEL CREATION
# =============================================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nDEVICE")
print("-" * 80)

print(
    "Device:",
    device
)


model = MultimodalFusionModel(
    cxr_input_dim=768,
    cxr_projection_dim=256,
    genomic_input_dim=128,
    genomic_projection_dim=128,
    fusion_hidden_dim=128,
    num_classes=2,
    dropout=0.20
).to(device)


# =============================================================================
# 17. LOAD SAVED BEST CHECKPOINT
# =============================================================================

print("\nBEST CHECKPOINT")
print("-" * 80)

checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location=device,
    weights_only=False
)

checkpoint_epoch = int(
    checkpoint[
        "epoch"
    ]
)

checkpoint_auc = float(
    checkpoint[
        "best_validation_roc_auc"
    ]
)

print(
    "Checkpoint epoch:",
    checkpoint_epoch
)

print(
    "Checkpoint validation ROC-AUC:",
    f"{checkpoint_auc:.6f}"
)


if checkpoint_epoch != expected_best_epoch:

    raise RuntimeError(
        "Checkpoint epoch does not match Cell 4 summary."
    )

if not np.isclose(
    checkpoint_auc,
    expected_best_auc,
    atol=1e-10
):

    raise RuntimeError(
        "Checkpoint ROC-AUC does not match Cell 4 summary."
    )


missing_keys, unexpected_keys = (
    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ],
        strict=False
    )
)

if missing_keys:

    raise RuntimeError(
        f"Missing checkpoint keys: {missing_keys}"
    )

if unexpected_keys:

    raise RuntimeError(
        f"Unexpected checkpoint keys: {unexpected_keys}"
    )


model.eval()

print(
    "Checkpoint load: PASS"
)

print(
    "Frozen checkpoint epoch:",
    checkpoint_epoch
)


# =============================================================================
# 18. VALIDATION INFERENCE
# =============================================================================

print("\n")
print("=" * 80)
print("VALIDATION-ONLY CHECKPOINT INFERENCE")
print("=" * 80)

BATCH_SIZE = 32

validation_probabilities = []
validation_logits = []

with torch.no_grad():

    for start in range(
        0,
        len(validation_cxr),
        BATCH_SIZE
    ):

        end = min(
            start + BATCH_SIZE,
            len(validation_cxr)
        )

        cxr_batch = torch.from_numpy(
            validation_cxr[
                start:end
            ]
        ).to(device)

        genomic_batch = torch.from_numpy(
            validation_genomic[
                start:end
            ]
        ).to(device)

        logits = model(
            cxr_batch,
            genomic_batch
        )

        if not torch.isfinite(
            logits
        ).all():

            raise RuntimeError(
                "Non-finite validation logits detected."
            )

        probabilities = torch.softmax(
            logits,
            dim=1
        )[:, 1]

        validation_logits.extend(
            logits.cpu()
            .numpy()
            .tolist()
        )

        validation_probabilities.extend(
            probabilities.cpu()
            .numpy()
            .tolist()
        )


validation_probabilities = np.asarray(
    validation_probabilities,
    dtype=np.float64
)

validation_logits = np.asarray(
    validation_logits,
    dtype=np.float64
)


if len(
    validation_probabilities
) != 395:

    raise RuntimeError(
        "Validation prediction count is not 395."
    )

print(
    "Validation predictions:",
    len(validation_probabilities)
)


# =============================================================================
# 19. FIXED 0.5 DECISION THRESHOLD
# =============================================================================
#
# IMPORTANT:
# The threshold is NOT tuned here.
# We simply reproduce the standard 0.5 decision threshold used during Cell 4.
# =============================================================================

VALIDATION_THRESHOLD = 0.5

validation_predictions = (
    validation_probabilities
    >=
    VALIDATION_THRESHOLD
).astype(
    np.int64
)


# =============================================================================
# 20. VALIDATION METRICS
# =============================================================================

validation_accuracy = accuracy_score(
    validation_targets,
    validation_predictions
)

validation_balanced_accuracy = (
    balanced_accuracy_score(
        validation_targets,
        validation_predictions
    )
)

validation_precision = precision_score(
    validation_targets,
    validation_predictions,
    zero_division=0
)

validation_sensitivity = recall_score(
    validation_targets,
    validation_predictions,
    zero_division=0
)

validation_f1 = f1_score(
    validation_targets,
    validation_predictions,
    zero_division=0
)

validation_specificity = (
    None
)

tn, fp, fn, tp = confusion_matrix(
    validation_targets,
    validation_predictions,
    labels=[0, 1]
).ravel()

if (
    tn + fp
) > 0:

    validation_specificity = (
        tn
        /
        (
            tn + fp
        )
    )

validation_roc_auc = roc_auc_score(
    validation_targets,
    validation_probabilities
)

validation_pr_auc = (
    average_precision_score(
        validation_targets,
        validation_probabilities
    )
)


# =============================================================================
# 21. CONFUSION MATRIX
# =============================================================================

cm = np.array(
    [
        [tn, fp],
        [fn, tp]
    ],
    dtype=np.int64
)


print("\n")
print("=" * 80)
print("INDEPENDENT VALIDATION RESULTS")
print("=" * 80)

print(
    f"Accuracy            : {validation_accuracy:.6f}"
)

print(
    f"Balanced Accuracy   : {validation_balanced_accuracy:.6f}"
)

print(
    f"Precision            : {validation_precision:.6f}"
)

print(
    f"Sensitivity / Recall : {validation_sensitivity:.6f}"
)

print(
    f"Specificity          : {validation_specificity:.6f}"
)

print(
    f"F1 Score             : {validation_f1:.6f}"
)

print(
    f"ROC-AUC              : {validation_roc_auc:.6f}"
)

print(
    f"PR-AUC               : {validation_pr_auc:.6f}"
)


print("\nCONFUSION MATRIX")
print("-" * 80)

print(
    "                 Predicted"
)

print(
    "                 DS     DR"
)

print(
    f"Actual DS        {tn:4d}   {fp:4d}"
)

print(
    f"Actual DR        {fn:4d}   {tp:4d}"
)


# =============================================================================
# 22. CHECKPOINT REPRODUCTION AUDIT
# =============================================================================

print("\nCHECKPOINT REPRODUCTION AUDIT")
print("-" * 80)

auc_difference = abs(
    validation_roc_auc
    -
    expected_best_auc
)

print(
    "Cell 4 recorded ROC-AUC:",
    f"{expected_best_auc:.10f}"
)

print(
    "Cell 5 recalculated ROC-AUC:",
    f"{validation_roc_auc:.10f}"
)

print(
    "Absolute difference:",
    f"{auc_difference:.12f}"
)


if auc_difference > 1e-8:

    raise RuntimeError(
        "Independent validation ROC-AUC does not reproduce "
        "the Cell 4 checkpoint result."
    )


print(
    "ROC-AUC reproduction: PASS"
)


# =============================================================================
# 23. VALIDATION PREDICTION TABLE
# =============================================================================

validation_predictions_df = pd.DataFrame({

    "condition_id":
        validation_ids,

    "target_binary":
        validation_targets,

    "target_label":
        np.where(
            validation_targets == 1,
            "DR-TB",
            "DS-TB"
        ),

    "probability_DR":
        validation_probabilities,

    "predicted_binary":
        validation_predictions,

    "predicted_label":
        np.where(
            validation_predictions == 1,
            "DR-TB",
            "DS-TB"
        ),

})


validation_predictions_df.to_csv(
    VALIDATION_PREDICTIONS_FILE,
    index=False
)


# =============================================================================
# 24. METRICS FILE
# =============================================================================

validation_metrics = {

    "checkpoint_epoch":
        checkpoint_epoch,

    "checkpoint_validation_roc_auc":
        checkpoint_auc,

    "independent_validation_accuracy":
        float(validation_accuracy),

    "independent_validation_balanced_accuracy":
        float(validation_balanced_accuracy),

    "independent_validation_precision":
        float(validation_precision),

    "independent_validation_sensitivity":
        float(validation_sensitivity),

    "independent_validation_specificity":
        float(validation_specificity),

    "independent_validation_f1":
        float(validation_f1),

    "independent_validation_roc_auc":
        float(validation_roc_auc),

    "independent_validation_pr_auc":
        float(validation_pr_auc),

    "confusion_matrix": {

        "TN":
            int(tn),

        "FP":
            int(fp),

        "FN":
            int(fn),

        "TP":
            int(tp),

    },

    "threshold":
        VALIDATION_THRESHOLD,

    "threshold_tuned":
        False,

    "test_used":
        False,

}


with open(
    VALIDATION_METRICS_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        validation_metrics,
        f,
        indent=4
    )


# =============================================================================
# 25. CONFUSION MATRIX FILE
# =============================================================================

cm_df = pd.DataFrame(
    cm,
    index=[
        "Actual_DS",
        "Actual_DR"
    ],
    columns=[
        "Predicted_DS",
        "Predicted_DR"
    ]
)

cm_df.to_csv(
    VALIDATION_CONFUSION_MATRIX_FILE
)


# =============================================================================
# 26. FINAL CELL 5 SUMMARY
# =============================================================================

cell5_summary = {

    "status":
        "PASS",

    "checkpoint_epoch":
        checkpoint_epoch,

    "checkpoint_validation_roc_auc":
        checkpoint_auc,

    "independent_validation_roc_auc":
        float(validation_roc_auc),

    "roc_auc_difference":
        float(auc_difference),

    "validation_conditions":
        395,

    "validation_ds":
        141,

    "validation_dr":
        254,

    "validation_accuracy":
        float(validation_accuracy),

    "validation_balanced_accuracy":
        float(validation_balanced_accuracy),

    "validation_precision":
        float(validation_precision),

    "validation_sensitivity":
        float(validation_sensitivity),

    "validation_specificity":
        float(validation_specificity),

    "validation_f1":
        float(validation_f1),

    "validation_pr_auc":
        float(validation_pr_auc),

    "threshold":
        0.5,

    "threshold_tuned":
        False,

    "training_performed":
        False,

    "fine_tuning_performed":
        False,

    "test_data_loaded_for_prediction":
        False,

    "test_predictions_generated":
        False,

    "test_metrics_calculated":
        False,

    "checkpoint_modified":
        False,

    "checkpoint_selection_repeated":
        False,

    "best_checkpoint":
        str(
            BEST_CHECKPOINT
        ),

    "validation_predictions":
        str(
            VALIDATION_PREDICTIONS_FILE
        ),

    "validation_metrics":
        str(
            VALIDATION_METRICS_FILE
        ),

    "confusion_matrix":
        str(
            VALIDATION_CONFUSION_MATRIX_FILE
        ),

}


with open(
    CELL5_SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        cell5_summary,
        f,
        indent=4
    )


# =============================================================================
# 27. FINAL STATUS
# =============================================================================

print("\n")
print("=" * 80)
print("CELL 5 STATUS: PASS")
print("=" * 80)

print(
    "Checkpoint epoch:",
    checkpoint_epoch
)

print(
    "Checkpoint validation ROC-AUC:",
    f"{checkpoint_auc:.6f}"
)

print(
    "Independent validation ROC-AUC:",
    f"{validation_roc_auc:.6f}"
)

print(
    "ROC-AUC reproduction difference:",
    f"{auc_difference:.12f}"
)

print(
    "Validation accuracy:",
    f"{validation_accuracy:.6f}"
)

print(
    "Validation F1:",
    f"{validation_f1:.6f}"
)

print(
    "Validation PR-AUC:",
    f"{validation_pr_auc:.6f}"
)

print(
    "Threshold:",
    "0.5"
)

print(
    "Threshold tuned:",
    "NO"
)

print(
    "Training performed:",
    "NO"
)

print(
    "Test predictions:",
    "NO"
)

print(
    "Test metrics:",
    "NOT CALCULATED"
)

print(
    "\nValidation predictions:"
)

print(
    VALIDATION_PREDICTIONS_FILE
)

print(
    "\nValidation metrics:"
)

print(
    VALIDATION_METRICS_FILE
)

print(
    "\nReview summary:"
)

print(
    CELL5_SUMMARY_FILE
)

print(
    "\nSTOP HERE."
)

print(
    "Proceed to Notebook 15 — Cell 6 only if STATUS = PASS."
)

NOTEBOOK 15 — CELL 5
FINAL MULTIMODAL BEST-CHECKPOINT VALIDATION REVIEW

PATH GATE
--------------------------------------------------------------------------------
Multimodal master               : PASS
Cell 1 summary                  : PASS
CXR embeddings                  : PASS
Genomic embeddings              : PASS
Targets                         : PASS
Condition IDs                   : PASS
Splits                          : PASS
Cell 2 summary                  : PASS
Best fusion checkpoint          : PASS
Cell 4 summary                  : PASS
Cell 4 history                  : PASS

PREVIOUS CELL STATUS
--------------------------------------------------------------------------------
Cell 1: PASS
Cell 2: PASS
Cell 4: PASS

CELL 4 TRAINING HISTORY AUDIT
--------------------------------------------------------------------------------
Epochs recorded: 40
Best history epoch: 1
Best history validation ROC-AUC: 0.963255
Summary best epoch: 1
Summary best validation ROC-AUC: 0.963255
Cell 

In [6]:
# =============================================================================
# NOTEBOOK 15 — CELL 6
# FINAL FROZEN MULTIMODAL CXR + GENOMIC TEST EVALUATION
#
# OFFICIAL FINAL TEST EVALUATION
#
# Fusion checkpoint : Epoch 1
# Selection metric  : Validation ROC-AUC
# Test threshold    : Fixed 0.5
#
# Training                 : NO
# Checkpoint selection     : NO
# Threshold tuning         : NO
# Test used before Cell 6  : NO
# =============================================================================

from pathlib import Path
import json
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

print("=" * 80)
print("NOTEBOOK 15 — CELL 6")
print("FINAL FROZEN MULTIMODAL CXR + GENOMIC TEST EVALUATION")
print("=" * 80)


# =============================================================================
# 1. REPRODUCIBILITY
# =============================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# =============================================================================
# 2. PROJECT PATHS
# =============================================================================

PROJECT_ROOT = Path(
    r"C:\TBP\Metadata"
)

FINAL_ROOT = (
    PROJECT_ROOT
    / "Step_3B_8_CXR_CoAtNet_Preparation"
    / "QC"
    / "Clean_PSPNet_CXR_Cohort"
    / "FINAL_FROZEN_CLEAN_COHORT"
)

MULTIMODAL_ROOT = (
    FINAL_ROOT
    / "Multimodal_15"
)

CELL1_ROOT = (
    MULTIMODAL_ROOT
    / "Cell1_Alignment_Audit"
)

CELL2_ROOT = (
    MULTIMODAL_ROOT
    / "Cell2_Frozen_Embedding_Extraction"
)

CELL4_ROOT = (
    MULTIMODAL_ROOT
    / "Cell4_Fusion_Training"
)

CELL5_ROOT = (
    MULTIMODAL_ROOT
    / "Cell5_Best_Checkpoint_Validation_Review"
)

CELL6_ROOT = (
    MULTIMODAL_ROOT
    / "Cell6_Frozen_Test_Evaluation"
)

CELL6_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 3. INPUT FILES
# =============================================================================

MULTIMODAL_MASTER = (
    CELL1_ROOT
    / "Notebook15_Cell1_FINAL_1976_Multimodal_Master.csv"
)

CELL1_SUMMARY = (
    CELL1_ROOT
    / "Notebook15_Cell1_Multimodal_Alignment_Audit_Summary.json"
)

CXR_EMBEDDINGS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_CXR_Embeddings_768D.npy"
)

GENOMIC_EMBEDDINGS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_Genomic_Embeddings_128D.npy"
)

TARGETS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_Targets.npy"
)

CONDITION_IDS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_Condition_IDs.npy"
)

SPLITS_FILE = (
    CELL2_ROOT
    / "Notebook15_Cell2_FINAL_1976_Splits.npy"
)

CELL2_SUMMARY = (
    CELL2_ROOT
    / "Notebook15_Cell2_Frozen_Embedding_Extraction_Summary.json"
)

BEST_CHECKPOINT = (
    CELL4_ROOT
    / "checkpoints"
    / "Notebook15_Cell4_Best_Multimodal_Validation_ROC_AUC.pth"
)

CELL4_SUMMARY = (
    CELL4_ROOT
    / "Notebook15_Cell4_Multimodal_Training_Summary.json"
)

CELL5_SUMMARY = (
    CELL5_ROOT
    / "Notebook15_Cell5_Best_Checkpoint_Validation_Review_Summary.json"
)

CELL5_METRICS = (
    CELL5_ROOT
    / "Notebook15_Cell5_Best_Checkpoint_Validation_Metrics.json"
)


# =============================================================================
# 4. OUTPUT FILES
# =============================================================================

TEST_PREDICTIONS_FILE = (
    CELL6_ROOT
    / "Notebook15_Cell6_FINAL_Frozen_Test_Predictions.csv"
)

TEST_METRICS_FILE = (
    CELL6_ROOT
    / "Notebook15_Cell6_FINAL_Frozen_Test_Metrics.json"
)

TEST_CONFUSION_MATRIX_FILE = (
    CELL6_ROOT
    / "Notebook15_Cell6_FINAL_Frozen_Test_Confusion_Matrix.csv"
)

TEST_BOOTSTRAP_FILE = (
    CELL6_ROOT
    / "Notebook15_Cell6_FINAL_Frozen_Test_Bootstrap_2000.csv"
)

TEST_SUMMARY_FILE = (
    CELL6_ROOT
    / "Notebook15_Cell6_FINAL_Frozen_Test_Evaluation_Summary.json"
)


# =============================================================================
# 5. PATH GATE
# =============================================================================

print("\nPATH GATE")
print("-" * 80)

required_paths = {

    "Multimodal master":
        MULTIMODAL_MASTER,

    "Cell 1 summary":
        CELL1_SUMMARY,

    "CXR embeddings":
        CXR_EMBEDDINGS_FILE,

    "Genomic embeddings":
        GENOMIC_EMBEDDINGS_FILE,

    "Targets":
        TARGETS_FILE,

    "Condition IDs":
        CONDITION_IDS_FILE,

    "Splits":
        SPLITS_FILE,

    "Cell 2 summary":
        CELL2_SUMMARY,

    "Best fusion checkpoint":
        BEST_CHECKPOINT,

    "Cell 4 summary":
        CELL4_SUMMARY,

    "Cell 5 summary":
        CELL5_SUMMARY,

    "Cell 5 metrics":
        CELL5_METRICS,

}

for name, path in required_paths.items():

    if not path.exists():

        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

    print(
        f"{name:<32}: PASS"
    )


# =============================================================================
# 6. LOAD PREVIOUS SUMMARIES
# =============================================================================

with open(
    CELL1_SUMMARY,
    "r",
    encoding="utf-8"
) as f:

    cell1_summary = json.load(f)


with open(
    CELL2_SUMMARY,
    "r",
    encoding="utf-8"
) as f:

    cell2_summary = json.load(f)


with open(
    CELL4_SUMMARY,
    "r",
    encoding="utf-8"
) as f:

    cell4_summary = json.load(f)


with open(
    CELL5_SUMMARY,
    "r",
    encoding="utf-8"
) as f:

    cell5_summary = json.load(f)


with open(
    CELL5_METRICS,
    "r",
    encoding="utf-8"
) as f:

    cell5_metrics = json.load(f)


# =============================================================================
# 7. PREVIOUS CELL STATUS GATES
# =============================================================================

print("\nPREVIOUS CELL STATUS")
print("-" * 80)

if cell1_summary.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Cell 1 did not PASS."
    )

if cell2_summary.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Cell 2 did not PASS."
    )

if cell4_summary.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Cell 4 did not PASS."
    )

if cell5_summary.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Cell 5 did not PASS."
    )

print("Cell 1: PASS")
print("Cell 2: PASS")
print("Cell 4: PASS")
print("Cell 5: PASS")


# =============================================================================
# 8. CHECKPOINT SELECTION AUDIT
# =============================================================================

print("\nCHECKPOINT SELECTION AUDIT")
print("-" * 80)

expected_epoch = int(
    cell4_summary[
        "best_epoch"
    ]
)

expected_auc = float(
    cell4_summary[
        "best_validation_roc_auc"
    ]
)

cell5_epoch = int(
    cell5_summary[
        "checkpoint_epoch"
    ]
)

cell5_auc = float(
    cell5_summary[
        "checkpoint_validation_roc_auc"
    ]
)

checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

checkpoint_epoch = int(
    checkpoint[
        "epoch"
    ]
)

checkpoint_auc = float(
    checkpoint[
        "best_validation_roc_auc"
    ]
)

print(
    "Cell 4 selected epoch:",
    expected_epoch
)

print(
    "Cell 5 verified epoch:",
    cell5_epoch
)

print(
    "Checkpoint epoch:",
    checkpoint_epoch
)

print(
    "Cell 4 validation ROC-AUC:",
    f"{expected_auc:.10f}"
)

print(
    "Cell 5 validation ROC-AUC:",
    f"{cell5_auc:.10f}"
)

print(
    "Checkpoint validation ROC-AUC:",
    f"{checkpoint_auc:.10f}"
)


if expected_epoch != 1:

    raise RuntimeError(
        "The frozen final multimodal checkpoint is not Epoch 1."
    )

if cell5_epoch != expected_epoch:

    raise RuntimeError(
        "Cell 5 checkpoint epoch does not match Cell 4."
    )

if checkpoint_epoch != expected_epoch:

    raise RuntimeError(
        "Checkpoint epoch does not match selected epoch."
    )

if not np.isclose(
    expected_auc,
    cell5_auc,
    atol=1e-10
):

    raise RuntimeError(
        "Cell 4 and Cell 5 validation ROC-AUC mismatch."
    )

if not np.isclose(
    expected_auc,
    checkpoint_auc,
    atol=1e-10
):

    raise RuntimeError(
        "Checkpoint ROC-AUC mismatch."
    )

print(
    "Frozen checkpoint identity: PASS"
)


# =============================================================================
# 9. TEST ISOLATION AUDIT BEFORE FIRST TEST INFERENCE
# =============================================================================

print("\nPRE-TEST ISOLATION AUDIT")
print("-" * 80)

print(
    "Training using test data:",
    "NO"
)

print(
    "Checkpoint selection using test:",
    "NO"
)

print(
    "Threshold tuning using test:",
    "NO"
)

print(
    "Test predictions before Cell 6:",
    "NO"
)

print(
    "Pre-test isolation: PASS"
)


# =============================================================================
# 10. LOAD EMBEDDINGS AND FROZEN SPLIT
# =============================================================================

cxr_embeddings = np.load(
    CXR_EMBEDDINGS_FILE
)

genomic_embeddings = np.load(
    GENOMIC_EMBEDDINGS_FILE
)

targets = np.load(
    TARGETS_FILE
)

condition_ids = np.load(
    CONDITION_IDS_FILE,
    allow_pickle=True
).astype(str)

splits = np.load(
    SPLITS_FILE,
    allow_pickle=True
).astype(str)


# =============================================================================
# 11. FULL DATASET SHAPE GATE
# =============================================================================

print("\nFROZEN MULTIMODAL DATASET")
print("-" * 80)

print(
    "CXR embeddings:",
    cxr_embeddings.shape
)

print(
    "Genomic embeddings:",
    genomic_embeddings.shape
)

print(
    "Targets:",
    targets.shape
)

print(
    "Condition IDs:",
    condition_ids.shape
)

print(
    "Splits:",
    splits.shape
)


if cxr_embeddings.shape != (
    1976,
    768
):

    raise RuntimeError(
        "CXR embedding shape mismatch."
    )

if genomic_embeddings.shape != (
    1976,
    128
):

    raise RuntimeError(
        "Genomic embedding shape mismatch."
    )

if targets.shape != (
    1976,
):

    raise RuntimeError(
        "Target shape mismatch."
    )

if condition_ids.shape != (
    1976,
):

    raise RuntimeError(
        "Condition ID shape mismatch."
    )

if splits.shape != (
    1976,
):

    raise RuntimeError(
        "Split shape mismatch."
    )

if len(
    set(condition_ids)
) != 1976:

    raise RuntimeError(
        "Condition IDs are not unique."
    )

print(
    "Frozen dataset shape gate: PASS"
)


# =============================================================================
# 12. TEST SPLIT EXTRACTION
# =============================================================================

test_indices = np.where(
    splits == "test"
)[0]

print("\nFROZEN TEST COHORT")
print("-" * 80)

print(
    "Test conditions:",
    len(test_indices)
)

if len(
    test_indices
) != 396:

    raise RuntimeError(
        "Expected exactly 396 frozen test conditions."
    )


test_cxr = (
    cxr_embeddings[
        test_indices
    ]
    .astype(
        np.float32,
        copy=False
    )
)

test_genomic = (
    genomic_embeddings[
        test_indices
    ]
    .astype(
        np.float32,
        copy=False
    )
)

test_targets = (
    targets[
        test_indices
    ]
    .astype(
        np.int64,
        copy=False
    )
)

test_ids = (
    condition_ids[
        test_indices
    ]
)


# =============================================================================
# 13. TEST TARGET AUDIT
# =============================================================================

test_ds = int(
    (
        test_targets
        ==
        0
    ).sum()
)

test_dr = int(
    (
        test_targets
        ==
        1
    ).sum()
)

print(
    "Test DS:",
    test_ds
)

print(
    "Test DR:",
    test_dr
)

if test_ds != 141:

    raise RuntimeError(
        "Frozen test DS count mismatch."
    )

if test_dr != 255:

    raise RuntimeError(
        "Frozen test DR count mismatch."
    )

print(
    "Frozen test target distribution: PASS"
)


# =============================================================================
# 14. VERIFY TEST CONDITION IDs ARE DISJOINT
# =============================================================================

train_ids = set(
    condition_ids[
        splits == "train"
    ]
)

validation_ids = set(
    condition_ids[
        splits == "validation"
    ]
)

test_id_set = set(
    test_ids
)

print("\nTEST CONDITION ISOLATION")
print("-" * 80)

print(
    "Train ∩ Test:",
    len(
        train_ids
        &
        test_id_set
    )
)

print(
    "Validation ∩ Test:",
    len(
        validation_ids
        &
        test_id_set
    )
)

if train_ids & test_id_set:

    raise RuntimeError(
        "Test condition overlap with training detected."
    )

if validation_ids & test_id_set:

    raise RuntimeError(
        "Test condition overlap with validation detected."
    )

print(
    "Test condition isolation: PASS"
)


# =============================================================================
# 15. MODEL ARCHITECTURE
# =============================================================================

class MultimodalFusionModel(
    nn.Module
):

    def __init__(
        self,
        cxr_input_dim=768,
        cxr_projection_dim=256,
        genomic_input_dim=128,
        genomic_projection_dim=128,
        fusion_hidden_dim=128,
        num_classes=2,
        dropout=0.20
    ):

        super().__init__()

        self.cxr_projection = nn.Sequential(

            nn.Linear(
                cxr_input_dim,
                cxr_projection_dim
            ),

            nn.LayerNorm(
                cxr_projection_dim
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            )
        )

        self.genomic_projection = nn.Sequential(

            nn.Linear(
                genomic_input_dim,
                genomic_projection_dim
            ),

            nn.LayerNorm(
                genomic_projection_dim
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            )
        )

        self.fusion = nn.Sequential(

            nn.Linear(
                cxr_projection_dim
                +
                genomic_projection_dim,
                fusion_hidden_dim
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                fusion_hidden_dim,
                num_classes
            )
        )

    def forward(
        self,
        cxr_embedding,
        genomic_embedding
    ):

        cxr_representation = (
            self.cxr_projection(
                cxr_embedding
            )
        )

        genomic_representation = (
            self.genomic_projection(
                genomic_embedding
            )
        )

        fused_representation = torch.cat(
            [
                cxr_representation,
                genomic_representation
            ],
            dim=1
        )

        logits = self.fusion(
            fused_representation
        )

        return logits


# =============================================================================
# 16. DEVICE AND MODEL
# =============================================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nDEVICE")
print("-" * 80)

print(
    "Device:",
    device
)

model = MultimodalFusionModel(
    cxr_input_dim=768,
    cxr_projection_dim=256,
    genomic_input_dim=128,
    genomic_projection_dim=128,
    fusion_hidden_dim=128,
    num_classes=2,
    dropout=0.20
).to(device)


# =============================================================================
# 17. CHECKPOINT LOAD
# =============================================================================

loaded_checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location=device,
    weights_only=False
)

missing_keys, unexpected_keys = (
    model.load_state_dict(
        loaded_checkpoint[
            "model_state_dict"
        ],
        strict=False
    )
)

if missing_keys:

    raise RuntimeError(
        f"Missing checkpoint keys: {missing_keys}"
    )

if unexpected_keys:

    raise RuntimeError(
        f"Unexpected checkpoint keys: {unexpected_keys}"
    )

model.eval()

print(
    "Checkpoint loaded: PASS"
)

print(
    "Checkpoint epoch:",
    loaded_checkpoint[
        "epoch"
    ]
)


# =============================================================================
# 18. OFFICIAL TEST INFERENCE
# =============================================================================
#
# From this point onward the frozen test set is evaluated.
#
# No parameter updates.
# No checkpoint selection.
# No threshold tuning.
# =============================================================================

print("\n")
print("=" * 80)
print("STARTING OFFICIAL FROZEN TEST INFERENCE")
print("=" * 80)

TEST_THRESHOLD = 0.5
BATCH_SIZE = 32

test_probabilities = []
test_logits = []

with torch.no_grad():

    for start in range(
        0,
        len(test_cxr),
        BATCH_SIZE
    ):

        end = min(
            start + BATCH_SIZE,
            len(test_cxr)
        )

        cxr_batch = torch.from_numpy(
            test_cxr[
                start:end
            ]
        ).to(device)

        genomic_batch = torch.from_numpy(
            test_genomic[
                start:end
            ]
        ).to(device)

        logits = model(
            cxr_batch,
            genomic_batch
        )

        if not torch.isfinite(
            logits
        ).all():

            raise RuntimeError(
                "Non-finite test logits detected."
            )

        probabilities = torch.softmax(
            logits,
            dim=1
        )[:, 1]

        test_logits.extend(
            logits.cpu()
            .numpy()
            .tolist()
        )

        test_probabilities.extend(
            probabilities.cpu()
            .numpy()
            .tolist()
        )


test_probabilities = np.asarray(
    test_probabilities,
    dtype=np.float64
)

test_logits = np.asarray(
    test_logits,
    dtype=np.float64
)


if len(
    test_probabilities
) != 396:

    raise RuntimeError(
        "Expected 396 test predictions."
    )


print(
    "Test predictions:",
    len(test_probabilities)
)

print(
    "Prediction count: PASS"
)


# =============================================================================
# 19. FIXED THRESHOLD
# =============================================================================
#
# This is NOT tuned on the test set.
# It is fixed at 0.5 before evaluating the test predictions.
# =============================================================================

test_predictions = (
    test_probabilities
    >=
    TEST_THRESHOLD
).astype(
    np.int64
)


# =============================================================================
# 20. FINAL TEST METRICS
# =============================================================================

test_accuracy = accuracy_score(
    test_targets,
    test_predictions
)

test_balanced_accuracy = (
    balanced_accuracy_score(
        test_targets,
        test_predictions
    )
)

test_precision = precision_score(
    test_targets,
    test_predictions,
    zero_division=0
)

test_sensitivity = recall_score(
    test_targets,
    test_predictions,
    zero_division=0
)

test_f1 = f1_score(
    test_targets,
    test_predictions,
    zero_division=0
)

test_roc_auc = roc_auc_score(
    test_targets,
    test_probabilities
)

test_pr_auc = (
    average_precision_score(
        test_targets,
        test_probabilities
    )
)


# =============================================================================
# 21. CONFUSION MATRIX
# =============================================================================

tn, fp, fn, tp = confusion_matrix(
    test_targets,
    test_predictions,
    labels=[0, 1]
).ravel()

test_specificity = (
    tn
    /
    (
        tn + fp
    )
)


# =============================================================================
# 22. FINAL TEST RESULTS
# =============================================================================

print("\n")
print("=" * 80)
print("FINAL FROZEN MULTIMODAL TEST RESULTS")
print("=" * 80)

print(
    f"Accuracy            : {test_accuracy:.6f}"
)

print(
    f"Balanced Accuracy   : {test_balanced_accuracy:.6f}"
)

print(
    f"Precision            : {test_precision:.6f}"
)

print(
    f"Sensitivity / Recall : {test_sensitivity:.6f}"
)

print(
    f"Specificity          : {test_specificity:.6f}"
)

print(
    f"F1 Score             : {test_f1:.6f}"
)

print(
    f"ROC-AUC              : {test_roc_auc:.6f}"
)

print(
    f"PR-AUC               : {test_pr_auc:.6f}"
)


# =============================================================================
# 23. CONFUSION MATRIX
# =============================================================================

print("\nCONFUSION MATRIX")
print("-" * 80)

print(
    "                 Predicted"
)

print(
    "                 DS     DR"
)

print(
    f"Actual DS        {tn:4d}   {fp:4d}"
)

print(
    f"Actual DR        {fn:4d}   {tp:4d}"
)


# =============================================================================
# 24. PREDICTION DISTRIBUTION
# =============================================================================

predicted_ds = int(
    (
        test_predictions
        ==
        0
    ).sum()
)

predicted_dr = int(
    (
        test_predictions
        ==
        1
    ).sum()
)

print("\nPREDICTED DISTRIBUTION")
print("-" * 80)

print(
    "Predicted DS:",
    predicted_ds
)

print(
    "Predicted DR:",
    predicted_dr
)

print(
    "Actual DS:",
    test_ds
)

print(
    "Actual DR:",
    test_dr
)


# =============================================================================
# 25. CLASSIFICATION REPORT
# =============================================================================

from sklearn.metrics import classification_report

print("\nCLASSIFICATION REPORT")
print("-" * 80)

print(
    classification_report(
        test_targets,
        test_predictions,
        labels=[0, 1],
        target_names=[
            "DS-TB",
            "DR-TB"
        ],
        digits=4,
        zero_division=0
    )
)


# =============================================================================
# 26. BOOTSTRAP CONFIDENCE INTERVALS
# =============================================================================
#
# Bootstrap is performed AFTER the model and threshold are frozen.
# It does NOT alter the model, checkpoint, or threshold.
#
# 2000 bootstrap resamples.
# =============================================================================

print("\n")
print("=" * 80)
print("BOOTSTRAP 95% CONFIDENCE INTERVALS")
print("=" * 80)

BOOTSTRAP_ITERATIONS = 2000

bootstrap_rng = np.random.default_rng(
    SEED
)

bootstrap_results = {

    "accuracy": [],
    "balanced_accuracy": [],
    "precision": [],
    "sensitivity": [],
    "specificity": [],
    "f1": [],
    "roc_auc": [],
    "pr_auc": [],

}


for iteration in range(
    BOOTSTRAP_ITERATIONS
):

    sample_indices = (
        bootstrap_rng.integers(
            0,
            len(test_targets),
            size=len(test_targets)
        )
    )

    y_true_bootstrap = (
        test_targets[
            sample_indices
        ]
    )

    y_prob_bootstrap = (
        test_probabilities[
            sample_indices
        ]
    )

    y_pred_bootstrap = (
        y_prob_bootstrap
        >=
        TEST_THRESHOLD
    ).astype(
        np.int64
    )

    # ROC-AUC and PR-AUC require both classes.
    if len(
        np.unique(
            y_true_bootstrap
        )
    ) < 2:

        continue

    bootstrap_results[
        "accuracy"
    ].append(
        accuracy_score(
            y_true_bootstrap,
            y_pred_bootstrap
        )
    )

    bootstrap_results[
        "balanced_accuracy"
    ].append(
        balanced_accuracy_score(
            y_true_bootstrap,
            y_pred_bootstrap
        )
    )

    bootstrap_results[
        "precision"
    ].append(
        precision_score(
            y_true_bootstrap,
            y_pred_bootstrap,
            zero_division=0
        )
    )

    bootstrap_results[
        "sensitivity"
    ].append(
        recall_score(
            y_true_bootstrap,
            y_pred_bootstrap,
            zero_division=0
        )
    )

    tn_b, fp_b, fn_b, tp_b = (
        confusion_matrix(
            y_true_bootstrap,
            y_pred_bootstrap,
            labels=[0, 1]
        ).ravel()
    )

    if (
        tn_b + fp_b
    ) > 0:

        specificity_b = (
            tn_b
            /
            (
                tn_b + fp_b
            )
        )

    else:

        specificity_b = np.nan

    bootstrap_results[
        "specificity"
    ].append(
        specificity_b
    )

    bootstrap_results[
        "f1"
    ].append(
        f1_score(
            y_true_bootstrap,
            y_pred_bootstrap,
            zero_division=0
        )
    )

    bootstrap_results[
        "roc_auc"
    ].append(
        roc_auc_score(
            y_true_bootstrap,
            y_prob_bootstrap
        )
    )

    bootstrap_results[
        "pr_auc"
    ].append(
        average_precision_score(
            y_true_bootstrap,
            y_prob_bootstrap
        )
    )


bootstrap_summary = {}

for metric_name, values in (
    bootstrap_results.items()
):

    values = np.asarray(
        values,
        dtype=np.float64
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:

        raise RuntimeError(
            f"No valid bootstrap values for {metric_name}."
        )

    lower = np.percentile(
        values,
        2.5
    )

    upper = np.percentile(
        values,
        97.5
    )

    bootstrap_summary[
        metric_name
    ] = {

        "point_estimate":
            float(
                {
                    "accuracy":
                        test_accuracy,

                    "balanced_accuracy":
                        test_balanced_accuracy,

                    "precision":
                        test_precision,

                    "sensitivity":
                        test_sensitivity,

                    "specificity":
                        test_specificity,

                    "f1":
                        test_f1,

                    "roc_auc":
                        test_roc_auc,

                    "pr_auc":
                        test_pr_auc,

                }[
                    metric_name
                ]
            ),

        "ci_lower_95":
            float(lower),

        "ci_upper_95":
            float(upper),

        "bootstrap_valid_samples":
            int(len(values)),

    }


print(
    f"Bootstrap iterations requested: "
    f"{BOOTSTRAP_ITERATIONS}"
)

print(
    "\nMetric                  Estimate      95% CI"
)

for metric_name, result in (
    bootstrap_summary.items()
):

    print(
        f"{metric_name:<23} "
        f"{result['point_estimate']:.4f}       "
        f"[{result['ci_lower_95']:.4f}, "
        f"{result['ci_upper_95']:.4f}]"
    )


# =============================================================================
# 27. SAVE TEST PREDICTIONS
# =============================================================================

test_predictions_df = pd.DataFrame({

    "condition_id":
        test_ids,

    "target_binary":
        test_targets,

    "target_label":
        np.where(
            test_targets == 1,
            "DR-TB",
            "DS-TB"
        ),

    "probability_DR":
        test_probabilities,

    "predicted_binary":
        test_predictions,

    "predicted_label":
        np.where(
            test_predictions == 1,
            "DR-TB",
            "DS-TB"
        ),

})


test_predictions_df.to_csv(
    TEST_PREDICTIONS_FILE,
    index=False
)


# =============================================================================
# 28. SAVE CONFUSION MATRIX
# =============================================================================

test_cm_df = pd.DataFrame(
    [
        [tn, fp],
        [fn, tp]
    ],
    index=[
        "Actual_DS",
        "Actual_DR"
    ],
    columns=[
        "Predicted_DS",
        "Predicted_DR"
    ]
)

test_cm_df.to_csv(
    TEST_CONFUSION_MATRIX_FILE
)


# =============================================================================
# 29. SAVE BOOTSTRAP RESULTS
# =============================================================================

bootstrap_rows = []

for metric_name, result in (
    bootstrap_summary.items()
):

    bootstrap_rows.append({

        "metric":
            metric_name,

        "point_estimate":
            result[
                "point_estimate"
            ],

        "ci_lower_95":
            result[
                "ci_lower_95"
            ],

        "ci_upper_95":
            result[
                "ci_upper_95"
            ],

        "bootstrap_valid_samples":
            result[
                "bootstrap_valid_samples"
            ],

    })


bootstrap_df = pd.DataFrame(
    bootstrap_rows
)

bootstrap_df.to_csv(
    TEST_BOOTSTRAP_FILE,
    index=False
)


# =============================================================================
# 30. SAVE TEST METRICS
# =============================================================================

test_metrics = {

    "evaluation_type":
        "FINAL_FROZEN_TEST_EVALUATION",

    "checkpoint_epoch":
        checkpoint_epoch,

    "checkpoint_validation_roc_auc":
        checkpoint_auc,

    "test_conditions":
        396,

    "test_ds":
        test_ds,

    "test_dr":
        test_dr,

    "threshold":
        TEST_THRESHOLD,

    "threshold_tuned_on_test":
        False,

    "accuracy":
        float(test_accuracy),

    "balanced_accuracy":
        float(test_balanced_accuracy),

    "precision":
        float(test_precision),

    "sensitivity":
        float(test_sensitivity),

    "specificity":
        float(test_specificity),

    "f1":
        float(test_f1),

    "roc_auc":
        float(test_roc_auc),

    "pr_auc":
        float(test_pr_auc),

    "confusion_matrix": {

        "TN":
            int(tn),

        "FP":
            int(fp),

        "FN":
            int(fn),

        "TP":
            int(tp),

    },

    "predicted_ds":
        predicted_ds,

    "predicted_dr":
        predicted_dr,

    "bootstrap_iterations":
        BOOTSTRAP_ITERATIONS,

    "bootstrap_summary":
        bootstrap_summary,

    "training_performed_in_cell":
        False,

    "checkpoint_selection_in_cell":
        False,

    "threshold_tuning_in_cell":
        False,

}


with open(
    TEST_METRICS_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        test_metrics,
        f,
        indent=4
    )


# =============================================================================
# 31. FINAL EVALUATION SUMMARY
# =============================================================================

final_summary = {

    "status":
        "PASS",

    "evaluation_type":
        "FINAL_FROZEN_TEST_EVALUATION",

    "checkpoint_epoch":
        checkpoint_epoch,

    "validation_checkpoint_roc_auc":
        checkpoint_auc,

    "test_conditions":
        396,

    "test_ds":
        test_ds,

    "test_dr":
        test_dr,

    "threshold":
        TEST_THRESHOLD,

    "threshold_tuned_on_test":
        False,

    "checkpoint_selected_on_test":
        False,

    "test_used_for_training":
        False,

    "accuracy":
        float(test_accuracy),

    "balanced_accuracy":
        float(test_balanced_accuracy),

    "precision":
        float(test_precision),

    "sensitivity":
        float(test_sensitivity),

    "specificity":
        float(test_specificity),

    "f1":
        float(test_f1),

    "roc_auc":
        float(test_roc_auc),

    "pr_auc":
        float(test_pr_auc),

    "confusion_matrix": {

        "TN":
            int(tn),

        "FP":
            int(fp),

        "FN":
            int(fn),

        "TP":
            int(tp),

    },

    "predicted_distribution": {

        "DS":
            predicted_ds,

        "DR":
            predicted_dr,

    },

    "bootstrap_iterations":
        BOOTSTRAP_ITERATIONS,

    "test_predictions_file":
        str(
            TEST_PREDICTIONS_FILE
        ),

    "test_metrics_file":
        str(
            TEST_METRICS_FILE
        ),

    "confusion_matrix_file":
        str(
            TEST_CONFUSION_MATRIX_FILE
        ),

    "bootstrap_file":
        str(
            TEST_BOOTSTRAP_FILE
        ),

}


with open(
    TEST_SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_summary,
        f,
        indent=4
    )


# =============================================================================
# 32. FINAL STATUS
# =============================================================================

print("\n")
print("=" * 80)
print("CELL 6 STATUS: PASS")
print("=" * 80)

print(
    "Checkpoint epoch:",
    checkpoint_epoch
)

print(
    "Test conditions:",
    len(test_targets)
)

print(
    "Test threshold:",
    TEST_THRESHOLD
)

print(
    "Threshold tuned using test:",
    "NO"
)

print(
    "Checkpoint selected using test:",
    "NO"
)

print(
    f"\nFINAL TEST ACCURACY: {test_accuracy:.6f}"
)

print(
    f"FINAL TEST BALANCED ACCURACY: "
    f"{test_balanced_accuracy:.6f}"
)

print(
    f"FINAL TEST PRECISION: {test_precision:.6f}"
)

print(
    f"FINAL TEST SENSITIVITY: {test_sensitivity:.6f}"
)

print(
    f"FINAL TEST SPECIFICITY: {test_specificity:.6f}"
)

print(
    f"FINAL TEST F1: {test_f1:.6f}"
)

print(
    f"FINAL TEST ROC-AUC: {test_roc_auc:.6f}"
)

print(
    f"FINAL TEST PR-AUC: {test_pr_auc:.6f}"
)

print(
    "\nCONFUSION MATRIX:"
)

print(
    f"TN={tn}, FP={fp}, FN={fn}, TP={tp}"
)

print(
    "\nTest predictions:"
)

print(
    TEST_PREDICTIONS_FILE
)

print(
    "\nTest metrics:"
)

print(
    TEST_METRICS_FILE
)

print(
    "\nBootstrap results:"
)

print(
    TEST_BOOTSTRAP_FILE
)

print(
    "\nFinal summary:"
)

print(
    TEST_SUMMARY_FILE
)

print(
    "\nIMPORTANT:"
)

print(
    "The frozen test set has now been evaluated."
)

print(
    "Do NOT retrain, tune the threshold, or modify the checkpoint "
    "based on these test results."
)

print(
    "\nSTOP HERE."
)

print(
    "Proceed to post-test analysis only after reviewing these results."
)

NOTEBOOK 15 — CELL 6
FINAL FROZEN MULTIMODAL CXR + GENOMIC TEST EVALUATION

PATH GATE
--------------------------------------------------------------------------------
Multimodal master               : PASS
Cell 1 summary                  : PASS
CXR embeddings                  : PASS
Genomic embeddings              : PASS
Targets                         : PASS
Condition IDs                   : PASS
Splits                          : PASS
Cell 2 summary                  : PASS
Best fusion checkpoint          : PASS
Cell 4 summary                  : PASS
Cell 5 summary                  : PASS
Cell 5 metrics                  : PASS

PREVIOUS CELL STATUS
--------------------------------------------------------------------------------
Cell 1: PASS
Cell 2: PASS
Cell 4: PASS
Cell 5: PASS

CHECKPOINT SELECTION AUDIT
--------------------------------------------------------------------------------
Cell 4 selected epoch: 1
Cell 5 verified epoch: 1
Checkpoint epoch: 1
Cell 4 validation ROC-AUC: 0.96